# NB10 — Repeated Validation and cycle_index Ablation

**Dissertation:** Explainable and Trustworthy Multimodal Deep Learning for Predictive Maintenance of Industrial Assets  
**Student:** 2023AA05069 | AIMLCZG628T | BITS Pilani  
**Notebook role:** Repeated validation across three engine-level split configurations + `cycle_index` ablation  
**Environment:** Google Colab T4  
**Note:** I run this notebook in sections. The seed-42 reproduction gate in Section 7 must pass before I begin any new training with seeds 21 and 84.

---

## Purpose and frozen protocol

A key limitation of the mid-semester evaluation is that the reported fusion result was obtained using one engine-level train-validation split, with split seed 42. This notebook strengthens the evaluation by retaining the existing seed-42 result and training the same frozen model configurations on two additional engine-level splits, using seeds 21 and 84. The results are then summarised across the three split configurations.

Holding the model seed fixed at 42 across all splits reduces variation from weight initialisation and makes the comparison primarily reflect sensitivity to engine-level split composition.

The notebook is designed to include a targeted `cycle_index` ablation on the seed-42 split to examine the contribution of temporal position information to each model's performance.

| Parameter | Value |
|---|---|
| Dataset | NASA C-MAPSS FD001 |
| RUL cap | 125 cycles |
| Rolling window for derived features | 5 cycles |
| Sequence window | 30 cycles |
| Reference split seed | 42 (mid-semester) |
| Model seed | 42 (fixed across all splits) |
| Split seeds | [21, 42, 84] |
| Test set access | None — test set is not opened in this notebook |

**SPLIT_SEED** controls which engines go to training and which to validation.  
**MODEL_SEED** controls weight initialisation, dropout masks, and data shuffling. Fixing it at 42 across all split configurations ensures the comparison primarily reflects engine-level split composition rather than a mixture of split composition and weight-initialisation variation.

## Section 0 — Colab and Project Setup

I mount Google Drive, verify that the project root is accessible, check for a GPU, and create the output directories for final-phase artefacts. All output paths are under `reports/final_validation/` and `models/final_validation/` to keep final-phase results separate from mid-semester artefacts.

In [1]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Dissertation/Project/dissertation-rul-xai'
else:
    BASE = os.path.abspath('..')

# Verify project root
assert os.path.isdir(BASE), f'Project root not found: {BASE}'
assert os.path.isfile(f'{BASE}/data/processed/train_val_split_fd001.csv'), \
    'train_val_split_fd001.csv not found — check Drive path'

# Paths
RAW_DIR        = f'{BASE}/data/raw/CMAPSS'
PROCESSED_DIR  = f'{BASE}/data/processed'
ARRAY_DIR      = f'{BASE}/data/processed/multiview'
FV_BASE        = f'{BASE}/data/processed/final_validation'
MV_BASE        = f'{BASE}/models/final_validation'
RV_DIR         = f'{BASE}/reports/final_validation'

# Create final-phase output directories
for seed in [21, 42, 84]:
    os.makedirs(f'{FV_BASE}/split_seed_{seed}', exist_ok=True)
    os.makedirs(f'{MV_BASE}/split_seed_{seed}', exist_ok=True)
os.makedirs(RV_DIR, exist_ok=True)

# GPU check
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {len(gpus) > 0}')
if gpus:
    print(f'GPU device: {gpus[0]}')
else:
    print('WARNING: No GPU detected. DerivedOnlyMLP and MultiViewGRUFusion training will be slow.')

print(f'Project root: {BASE}')
print('Directories ready.')

Mounted at /content/drive
TensorFlow: 2.20.0
GPU available: True
GPU device: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
Project root: /content/drive/MyDrive/Dissertation/Project/dissertation-rul-xai
Directories ready.


## Section 1 — Environment Manifest

I record the versions of all key packages and the hardware configuration before running any computation. This makes it possible to reproduce the results later if library versions change. The manifest is saved to `reports/final_validation/experiment_environment.json`.

In [2]:
import json
import platform
import subprocess
import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb
import gc
from datetime import datetime, timezone

# Attempt deterministic ops (Colab T4 supports this from TF 2.8+)
deterministic_enabled = False
try:
    tf.config.experimental.enable_op_determinism()
    deterministic_enabled = True
except Exception as e:
    print(f'enable_op_determinism not available: {e}')

# GPU device name — prefer nvidia-smi for the actual hardware model
gpu_name = 'None'
gpu_model = 'None'
if gpus:
    try:
        gpu_model = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            text=True
        ).strip()
    except Exception:
        gpu_model = str(gpus[0])
    try:
        gpu_name = tf.test.gpu_device_name()
    except Exception:
        gpu_name = str(gpus[0])

env_manifest = {
    'recorded_at':         datetime.now(timezone.utc).isoformat(),
    'python':              platform.python_version(),
    'tensorflow':          tf.__version__,
    'numpy':               np.__version__,
    'pandas':              pd.__version__,
    'scikit_learn':        sklearn.__version__,
    'xgboost':             xgb.__version__,
    'gpu_available':       len(gpus) > 0,
    'gpu_device':          gpu_name,
    'gpu_model':           gpu_model,
    'deterministic_ops':   deterministic_enabled,
    'platform':            platform.platform(),
}

env_path = f'{RV_DIR}/experiment_environment.json'
with open(env_path, 'w') as f:
    json.dump(env_manifest, f, indent=2)

print('Environment manifest:')
for k, v in env_manifest.items():
    print(f'  {k}: {v}')
print(f'Saved to: {env_path}')

Environment manifest:
  recorded_at: 2026-07-27T00:38:02.265158+00:00
  python: 3.12.13
  tensorflow: 2.20.0
  numpy: 2.0.2
  pandas: 2.2.2
  scikit_learn: 1.6.1
  xgboost: 3.3.0
  gpu_available: True
  gpu_device: /device:GPU:0
  gpu_model: Tesla T4
  deterministic_ops: True
  platform: Linux-6.6.122+-x86_64-with-glibc2.35
Saved to: /content/drive/MyDrive/Dissertation/Project/dissertation-rul-xai/reports/final_validation/experiment_environment.json


## Section 2 — Frozen Experiment Configuration

All model architectures, training hyperparameters, and reference metrics from the mid-semester run are fixed here. I do not change any of these values during execution. The frozen configuration is also saved as a JSON file alongside the outputs produced by this notebook.

In [3]:
# ── Seeds ──────────────────────────────────────────────────────────────────
SPLIT_SEEDS          = [21, 42, 84]   # controls engine-cohort assignment
REFERENCE_SPLIT_SEED = 42             # the mid-semester seed; gate must reproduce this
MODEL_SEED           = 42             # fixed; controls weight init, dropout, shuffling

# ── Dataset parameters ─────────────────────────────────────────────────────
RUL_CAP       = 125
WINDOW_SIZE   = 30
ROLLING_WIN   = 5
STRIDE        = 1
N_TRAIN_ENG   = 80
N_VAL_ENG     = 20

# ── Feature sets (verified against NB03 output) ────────────────────────────
SENSOR_COLS = [
    'sensor_measurement_11', 'sensor_measurement_4',  'sensor_measurement_12',
    'sensor_measurement_7',  'sensor_measurement_15', 'sensor_measurement_21',
    'sensor_measurement_20', 'sensor_measurement_2',  'sensor_measurement_17',
    'sensor_measurement_3',  'sensor_measurement_8',  'sensor_measurement_13',
    'sensor_measurement_9',  'sensor_measurement_14',
]  # 14 variable sensors (Feature Set B)

METADATA_COLS = ['unit_number', 'time_in_cycles', 'RUL', 'RUL_capped']
TARGET_COL    = 'RUL_capped'

# ── XGBoost configuration (frozen from NB04) ───────────────────────────────
XGB_PARAMS = dict(
    n_estimators    = 300,
    learning_rate   = 0.05,
    max_depth       = 4,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    objective       = 'reg:squarederror',
    random_state    = MODEL_SEED,
    n_jobs          = -1,
)

# ── GRU configuration (frozen from NB05) ──────────────────────────────────
GRU_MAX_EPOCHS  = 30
GRU_BATCH       = 256
GRU_ES_PATIENCE = 10
GRU_LR_PATIENCE = 5
GRU_LR          = 0.001
GRU_LR_FACTOR   = 0.5
GRU_MIN_LR      = 1e-6
# Architecture: GRU(64, return_sequences=True) → GRU(32) → Dense(50, relu) → Dropout(0.2) → Dense(1)

# ── DerivedOnlyMLP / MultiViewGRUFusion configuration (frozen from NB06) ──
MLP_MAX_EPOCHS  = 60
MLP_BATCH       = 128
MLP_ES_PATIENCE = 8
MLP_LR_PATIENCE = 4
MLP_MIN_LR      = 1e-5
MLP_LR          = 0.001
MLP_LR_FACTOR   = 0.5
# DerivedOnlyMLP architecture: Dense(64, relu) → Dropout(0.2) → Dense(32, relu) → Dense(1)
# MultiViewGRUFusion:
#   Sequence branch: GRU(64) → Dropout(0.2)
#   Derived branch:  Dense(64, relu) → Dropout(0.2) → Dense(32, relu)
#   Fusion:          Concatenate → Dense(64, relu) → Dropout(0.2) → Dense(32, relu) → Dense(1)
# Optimiser: Adam; Loss: MSE; Prediction clipping: [0, RUL_CAP]

# ── Reference best epochs (seed 42, mid-semester) ─────────────────────────
REF_BEST_EPOCH = {
    'GRU':                22,
    'DerivedOnlyMLP':     60,   # hit maximum
    'MultiViewGRUFusion': 18,
}

# ── Reference metrics (seed 42, window-aligned validation) ────────────────
REF_METRICS = {
    'MultiViewGRUFusion': {'rmse': 12.0657, 'mae': 8.9406,  'r2': 0.9168},
    'XGBoost':            {'rmse': 12.4894, 'mae': 9.2675,  'r2': 0.9109},
    'DerivedOnlyMLP':     {'rmse': 13.1451, 'mae': 9.4204,  'r2': 0.9013},
    'GRU':                {'rmse': 13.1605, 'mae': 9.7182,  'r2': 0.9010},
}
METRIC_TOLERANCE = 0.005   # max acceptable absolute difference for RMSE gate check

# ── Execution flags ────────────────────────────────────────────────────────
OVERWRITE_EXISTING       = False   # set True only to force rerun
RUN_CYCLE_INDEX_ABLATION = True

# ── Derived feature name pattern (documented for auditability) ─────────────
# Pattern: {sensor}_rmean, {sensor}_rstd, {sensor}_delta for each of 14 sensors
# Plus: cycle_index (= time_in_cycles at prediction cycle)
# Total: 14×3 + 1 = 43 derived features
# Exact names are confirmed after compute_derived_features runs (Section 7)

# ── Save frozen config ─────────────────────────────────────────────────────
frozen_config = {
    'SPLIT_SEEDS': SPLIT_SEEDS, 'REFERENCE_SPLIT_SEED': REFERENCE_SPLIT_SEED,
    'MODEL_SEED': MODEL_SEED, 'RUL_CAP': RUL_CAP, 'WINDOW_SIZE': WINDOW_SIZE,
    'ROLLING_WIN': ROLLING_WIN, 'STRIDE': STRIDE,
    'N_TRAIN_ENG': N_TRAIN_ENG, 'N_VAL_ENG': N_VAL_ENG,
    'SENSOR_COLS': SENSOR_COLS,
    'prediction_clipping': [0, RUL_CAP],
    'XGB_PARAMS': XGB_PARAMS,
    'GRU': {
        'max_epochs': GRU_MAX_EPOCHS, 'batch': GRU_BATCH,
        'es_patience': GRU_ES_PATIENCE, 'lr_patience': GRU_LR_PATIENCE,
        'lr': GRU_LR, 'lr_factor': GRU_LR_FACTOR, 'min_lr': GRU_MIN_LR,
        'dropout': 0.2,
        'architecture': 'GRU(64,seq=True)->GRU(32)->Dense(50,relu)->Dropout(0.2)->Dense(1)',
        'optimiser': 'Adam', 'loss': 'mse',
    },
    'MLP': {
        'max_epochs': MLP_MAX_EPOCHS, 'batch': MLP_BATCH,
        'es_patience': MLP_ES_PATIENCE, 'lr_patience': MLP_LR_PATIENCE,
        'min_lr': MLP_MIN_LR, 'lr': MLP_LR, 'lr_factor': MLP_LR_FACTOR,
        'dropout': 0.2,
        'DerivedOnlyMLP_architecture': 'Dense(64,relu)->Dropout(0.2)->Dense(32,relu)->Dense(1)',
        'MultiViewGRUFusion_seq_branch': 'GRU(64)->Dropout(0.2)',
        'MultiViewGRUFusion_der_branch': 'Dense(64,relu)->Dropout(0.2)->Dense(32,relu)',
        'MultiViewGRUFusion_fusion': 'Concat->Dense(64,relu)->Dropout(0.2)->Dense(32,relu)->Dense(1)',
        'optimiser': 'Adam', 'loss': 'mse',
    },
    'REF_BEST_EPOCH': REF_BEST_EPOCH,
    'REF_METRICS': REF_METRICS,
    'METRIC_TOLERANCE': METRIC_TOLERANCE,
    'derived_feature_pattern': '{sensor}_rmean, {sensor}_rstd, {sensor}_delta (×14 sensors) + cycle_index',
    'OVERWRITE_EXISTING': OVERWRITE_EXISTING,
    'RUN_CYCLE_INDEX_ABLATION': RUN_CYCLE_INDEX_ABLATION,
}
config_path = f'{RV_DIR}/frozen_experiment_config.json'
with open(config_path, 'w') as f:
    json.dump(frozen_config, f, indent=2)
print(f'Frozen config saved to: {config_path}')
print(f'SPLIT_SEEDS: {SPLIT_SEEDS}  |  MODEL_SEED: {MODEL_SEED}  |  METRIC_TOLERANCE: {METRIC_TOLERANCE}')

Frozen config saved to: /content/drive/MyDrive/Dissertation/Project/dissertation-rul-xai/reports/final_validation/frozen_experiment_config.json
SPLIT_SEEDS: [21, 42, 84]  |  MODEL_SEED: 42  |  METRIC_TOLERANCE: 0.005


## Section 3 — Raw Data Loading

I load the raw FD001 training trajectories from the original C-MAPSS text file and compute the RUL label for each row. The test file is not loaded in this notebook — test evaluation is handled separately in NB11 after this notebook is complete and frozen.

In [4]:
INDEX_COLS  = ['unit_number', 'time_in_cycles']
OP_COLS     = [f'operational_setting_{i}' for i in range(1, 4)]
ALL_SENSOR_COLS = [f'sensor_measurement_{i}' for i in range(1, 22)]
ALL_COLS    = INDEX_COLS + OP_COLS + ALL_SENSOR_COLS

raw_train = pd.read_csv(
    f'{RAW_DIR}/train_FD001.txt', sep=r'\s+', header=None, names=ALL_COLS
)

# Compute and cap RUL
max_cycles = raw_train.groupby('unit_number')['time_in_cycles'].max().rename('max_cycle')
raw_train  = raw_train.join(max_cycles, on='unit_number')
raw_train['RUL']        = raw_train['max_cycle'] - raw_train['time_in_cycles']
raw_train['RUL_capped'] = raw_train['RUL'].clip(upper=RUL_CAP).astype(float)
raw_train.drop(columns=['max_cycle'], inplace=True)

all_units = sorted(raw_train['unit_number'].unique())
print(f'Raw training data loaded: {raw_train.shape}')
print(f'Total engines: {len(all_units)}')
print(f'RUL range (uncapped): [{raw_train["RUL"].min()}, {raw_train["RUL"].max()}]')
print(f'RUL range (capped):   [{raw_train["RUL_capped"].min()}, {raw_train["RUL_capped"].max()}]')

Raw training data loaded: (20631, 28)
Total engines: 100
RUL range (uncapped): [0, 361]
RUL range (capped):   [0.0, 125.0]


## Section 4 — Shared Preprocessing Functions

These functions implement the same preprocessing pipeline I used in NB03 and NB06. I define them once here and use them for all three splits, which ensures the pipeline is identical across seeds.

The key design decisions carried forward from the mid-semester work are:
- Engine-level splitting (no row-level random split)
- Scaler fitted on training engines only, then applied to validation
- Rolling features computed within each engine independently, with no cross-engine look-ahead
- `cycle_index` included as a derived feature; `normalized_cycle_age` explicitly excluded

In [5]:
import random
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score


def make_engine_split(all_units, n_train, split_seed):
    """Reproducible engine-level 80/20 split using split_seed only.
    Uses legacy np.random API to match the NB03 split exactly."""
    np.random.seed(split_seed)
    shuffled    = np.random.permutation(all_units)
    train_units = sorted(shuffled[:n_train].tolist())
    val_units   = sorted(shuffled[n_train:].tolist())
    return train_units, val_units


def compute_derived_features(df, sensor_features, rolling_window=ROLLING_WIN):
    """Per-engine rolling mean, std, delta, and `cycle_index`. No cross-engine leakage."""
    parts = []
    for unit, unit_df in df.groupby('unit_number'):
        unit_df = unit_df.sort_values('time_in_cycles').copy()
        for s in sensor_features:
            unit_df[f'{s}_rmean'] = (
                unit_df[s].rolling(rolling_window, min_periods=1).mean()
            )
            unit_df[f'{s}_rstd'] = (
                unit_df[s].rolling(rolling_window, min_periods=1).std().fillna(0)
            )
            unit_df[f'{s}_delta'] = unit_df[s] - unit_df[s].iloc[0]
        unit_df['cycle_index'] = unit_df['time_in_cycles']
        parts.append(unit_df)
    return pd.concat(parts, ignore_index=True)


def fit_and_apply_scalers(train_df, val_df, feature_set_b, feature_set_c):
    """Fit scalers on training split only; apply to val. Returns scaled copies."""
    scaler_b = StandardScaler().fit(train_df[feature_set_b])
    scaler_c = StandardScaler().fit(train_df[feature_set_c])

    train_b = train_df.copy()
    val_b   = val_df.copy()
    train_c = train_df.copy()
    val_c   = val_df.copy()

    train_b[feature_set_b] = scaler_b.transform(train_df[feature_set_b])
    val_b[feature_set_b]   = scaler_b.transform(val_df[feature_set_b])
    train_c[feature_set_c] = scaler_c.transform(train_df[feature_set_c])
    val_c[feature_set_c]   = scaler_c.transform(val_df[feature_set_c])

    return train_b, val_b, train_c, val_c, scaler_b, scaler_c


def create_sequence_windows(df, sensor_cols, target_col, window_size=WINDOW_SIZE, stride=STRIDE):
    """Sliding 30-cycle windows of raw sensor sequence. Windows stay within engine."""
    X, y, meta = [], [], []
    for unit, unit_df in df.groupby('unit_number'):
        unit_df = unit_df.sort_values('time_in_cycles').reset_index(drop=True)
        feats   = unit_df[sensor_cols].values.astype(np.float32)
        targets = unit_df[target_col].values.astype(np.float32)
        cycles  = unit_df['time_in_cycles'].values
        raw_rul = unit_df['RUL'].values
        if len(unit_df) < window_size:
            continue
        for end in range(window_size - 1, len(unit_df), stride):
            start = end - window_size + 1
            X.append(feats[start:end + 1])
            y.append(targets[end])
            meta.append({'unit_number': unit, 'time_in_cycles': cycles[end],
                         'RUL': raw_rul[end], 'RUL_capped': targets[end]})
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32), pd.DataFrame(meta)


def create_multiview_windows(b_df, c_df, sensor_cols, derived_cols, target_col,
                              window_size=WINDOW_SIZE, stride=STRIDE):
    """Paired sequence + derived-feature windows. Both views share the same prediction cycle."""
    c_lookup = c_df.set_index(['unit_number', 'time_in_cycles'])
    X_seq, X_der, y, meta = [], [], [], []
    for unit, unit_df in b_df.groupby('unit_number'):
        unit_df = unit_df.sort_values('time_in_cycles').reset_index(drop=True)
        feats   = unit_df[sensor_cols].values.astype(np.float32)
        targets = unit_df[target_col].values.astype(np.float32)
        cycles  = unit_df['time_in_cycles'].values
        raw_rul = unit_df['RUL'].values
        if len(unit_df) < window_size:
            continue
        for end in range(window_size - 1, len(unit_df), stride):
            start = end - window_size + 1
            cycle = cycles[end]
            key   = (unit, cycle)
            if key not in c_lookup.index:
                continue
            X_seq.append(feats[start:end + 1])
            X_der.append(c_lookup.loc[key, derived_cols].values.astype(np.float32))
            y.append(targets[end])
            meta.append({'unit_number': unit, 'time_in_cycles': cycle,
                         'RUL': raw_rul[end], 'RUL_capped': targets[end]})
    return (np.array(X_seq, dtype=np.float32), np.array(X_der, dtype=np.float32),
            np.array(y, dtype=np.float32), pd.DataFrame(meta))


def evaluate_predictions(y_true, y_pred):
    """RMSE, MAE, R² with predictions clipped to [0, RUL_CAP].
    Returns full-precision floats; round only at display/reporting time."""
    y_pred = np.clip(np.asarray(y_pred, dtype=np.float64).reshape(-1), 0, RUL_CAP)
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    return {
        'rmse': float(root_mean_squared_error(y_true, y_pred)),
        'mae':  float(mean_absolute_error(y_true, y_pred)),
        'r2':   float(r2_score(y_true, y_pred)),
    }


def compute_per_engine_metrics(y_true_arr, y_pred_arr, meta_df):
    """Per-engine RMSE and MAE from window-level predictions and metadata.
    Full-precision values; do not round here — round at reporting time."""
    rows = []
    for unit in meta_df['unit_number'].unique():
        mask = (meta_df['unit_number'] == unit).values
        yt   = np.asarray(y_true_arr[mask], dtype=np.float64)
        yp   = np.clip(np.asarray(y_pred_arr[mask], dtype=np.float64), 0, RUL_CAP)
        rows.append({
            'unit_number': unit,
            'n_windows':   int(mask.sum()),
            'engine_rmse': float(root_mean_squared_error(yt, yp)),
            'engine_mae':  float(mean_absolute_error(yt, yp)),
            'mean_error':  float((yp - yt).mean()),
        })
    return pd.DataFrame(rows).sort_values('unit_number').reset_index(drop=True)


print('Shared preprocessing functions defined.')

Shared preprocessing functions defined.


## Section 5 — Leakage Assertions

Before I train any model on a new split, I run a set of assertions to confirm that the preprocessing is leakage-free. If any check fails, execution stops immediately with an informative error message. I do not proceed past a failed assertion.

In [6]:
def run_leakage_assertions(train_units, val_units, train_c_df, val_c_df,
                            X_train_seq, X_val_seq, X_train_der, X_val_der,
                            y_train, y_val, train_meta, val_meta,
                            feature_set_b, derived_cols):

    # 1. Correct engine counts
    assert len(train_units) == N_TRAIN_ENG, \
        f'Expected {N_TRAIN_ENG} train engines, got {len(train_units)}'
    assert len(val_units) == N_VAL_ENG, \
        f'Expected {N_VAL_ENG} val engines, got {len(val_units)}'

    # 2. No overlap between train and val engines
    overlap = set(train_units) & set(val_units)
    assert len(overlap) == 0, f'Train/val engine overlap: {overlap}'

    # 3. All 100 engines accounted for
    assert len(set(train_units) | set(val_units)) == 100, \
        'Train + val engines do not sum to 100'

    # 4. Val engines in val data only
    train_eng_in_meta = set(train_meta['unit_number'].unique())
    val_eng_in_meta   = set(val_meta['unit_number'].unique())
    assert train_eng_in_meta == set(train_units), 'Train meta contains unexpected engines'
    assert val_eng_in_meta   == set(val_units),   'Val meta contains unexpected engines'
    assert len(train_eng_in_meta & val_eng_in_meta) == 0, \
        'Engine appears in both train and val metadata'

    # 5. normalized_cycle_age must not appear in model inputs
    all_input_cols = set(feature_set_b) | set(derived_cols)
    assert 'normalized_cycle_age' not in all_input_cols, \
        'normalized_cycle_age found in model inputs — leakage risk'

    # 6. cycle_index must be present
    assert 'cycle_index' in derived_cols, '`cycle_index` missing from derived features'

    # 7. Target and metadata not in model inputs
    for forbidden in METADATA_COLS:
        assert forbidden not in all_input_cols, \
            f'Metadata/target column in model inputs: {forbidden}'

    # 8. Array shapes consistent
    assert X_train_seq.shape[0] == len(y_train) == len(train_meta)
    assert X_val_seq.shape[0]   == len(y_val)   == len(val_meta)
    assert X_train_seq.shape[1] == WINDOW_SIZE
    assert X_train_seq.shape[2] == len(feature_set_b)
    assert X_train_der.shape[1] == len(derived_cols)

    # 9. No NaN in arrays
    assert not np.isnan(X_train_seq).any(), 'NaN in X_train_seq'
    assert not np.isnan(X_train_der).any(), 'NaN in X_train_der'
    assert not np.isnan(X_val_seq).any(),   'NaN in X_val_seq'
    assert not np.isnan(X_val_der).any(),   'NaN in X_val_der'
    assert not np.isnan(y_train).any(),     'NaN in y_train'
    assert not np.isnan(y_val).any(),       'NaN in y_val'

    print('  All configured leakage-related checks passed.')


print('Leakage assertion function defined.')

Leakage assertion function defined.


## Section 6 — Frozen Model Builders

These are the exact model architectures I used in the mid-semester work — GRU from NB05 and DerivedOnlyMLP and MultiViewGRUFusion from NB06. I have not changed the architecture, layer sizes, or training configuration. The only deliberately varied experimental factor is the engine-level split.

Before each neural model is trained, the calling code must call `tf.keras.backend.clear_session()`, `gc.collect()`, and `tf.keras.utils.set_random_seed(MODEL_SEED)` separately — not once per split, but once per model. This is what makes the model-seed rule operational. The callback factory functions (`get_gru_callbacks`, `get_mlp_callbacks`) must also be called fresh for every training run; callback instances must not be reused.

In [7]:
from tensorflow.keras import layers, models, callbacks
import joblib
from xgboost import XGBRegressor


def build_gru(window_size, n_sensors):
    model = models.Sequential([
        layers.Input(shape=(window_size, n_sensors)),
        layers.GRU(64, return_sequences=True),
        layers.GRU(32),
        layers.Dense(50, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1),
    ], name='GRU')
    model.compile(optimizer=tf.keras.optimizers.Adam(GRU_LR), loss='mse', metrics=['mae'])
    return model


def build_derived_mlp(n_features):
    inp = layers.Input(shape=(n_features,), name='degradation_feature_view')
    x   = layers.Dense(64, activation='relu')(inp)
    x   = layers.Dropout(0.2)(x)
    x   = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(1, name='rul_prediction')(x)
    model = models.Model(inputs=inp, outputs=out, name='DerivedOnlyMLP')
    model.compile(optimizer=tf.keras.optimizers.Adam(MLP_LR), loss='mse', metrics=['mae'])
    return model


def build_multiview_gru(window_size, n_sensors, n_derived):
    seq_in  = layers.Input(shape=(window_size, n_sensors), name='sensor_sequence_view')
    seq_x   = layers.GRU(64, return_sequences=False, name='sensor_gru_encoder')(seq_in)
    seq_x   = layers.Dropout(0.2)(seq_x)

    der_in  = layers.Input(shape=(n_derived,), name='degradation_feature_view')
    der_x   = layers.Dense(64, activation='relu', name='degradation_dense_1')(der_in)
    der_x   = layers.Dropout(0.2)(der_x)
    der_x   = layers.Dense(32, activation='relu', name='degradation_dense_2')(der_x)

    fused   = layers.Concatenate(name='view_fusion')([seq_x, der_x])
    z       = layers.Dense(64, activation='relu', name='fusion_dense_1')(fused)
    z       = layers.Dropout(0.2)(z)
    z       = layers.Dense(32, activation='relu', name='fusion_dense_2')(z)
    out     = layers.Dense(1, name='rul_prediction')(z)

    model = models.Model(inputs=[seq_in, der_in], outputs=out, name='MultiViewGRUFusion')
    model.compile(optimizer=tf.keras.optimizers.Adam(MLP_LR), loss='mse', metrics=['mae'])
    return model


def get_gru_callbacks():
    return [
        callbacks.EarlyStopping(
            monitor='val_loss', patience=GRU_ES_PATIENCE,
            restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=GRU_LR_PATIENCE,
            min_lr=1e-6, verbose=1),
    ]


def get_mlp_callbacks():
    return [
        callbacks.EarlyStopping(
            monitor='val_loss', patience=MLP_ES_PATIENCE,
            restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=MLP_LR_PATIENCE,
            min_lr=MLP_MIN_LR, verbose=1),
    ]


print('Model builder functions defined.')

Model builder functions defined.


## Section 7 — Seed-42 Reproduction Gate

Before I train any new model, I need to confirm that my preprocessing pipeline in this notebook faithfully reproduces the mid-semester seed-42 split. This is important because any difference in the split function or feature construction would mean that the "seed-42" run in this notebook is not actually the same as the one I reported in my mid-semester results.

This gate reproduces the seed-42 engine split and preprocessing arrays and verifies the reported metrics from the stored prediction artefacts. It does not retrain the seed-42 neural models. New training is performed only for split configurations with seeds 21 and 84 using the frozen configurations.

The gate does the following:
1. Regenerates the seed-42 engine assignment using the `make_engine_split` function defined above
2. Compares the regenerated assignment against the saved `train_val_split_fd001.csv`
3. Rebuilds the multi-view arrays (both validation and training) and compares shapes, numerical values, and metadata against the saved NPZ and CSV files
4. Loads the saved prediction CSVs from the mid-semester run and recalculates RMSE, MAE, and R² for all four models
5. Checks that all four recalculated RMSE values are within the tolerance of the reference values; also checks MAE and R² agreement to four decimal places

What must match exactly are the engine assignments, array dimensions, metadata ordering, and cycle-level metadata values. Small floating-point differences in metrics are acceptable. If anything structural differs, I stop and investigate before running seeds 21 and 84.

In [8]:
print('=== SEED-42 REPRODUCTION GATE ===')
print()

# ── Step 1: Regenerate split ───────────────────────────────────────────────
regen_train_units, regen_val_units = make_engine_split(
    all_units, N_TRAIN_ENG, REFERENCE_SPLIT_SEED
)

# ── Step 2: Load saved split assignment ───────────────────────────────────
saved_split = pd.read_csv(f'{PROCESSED_DIR}/train_val_split_fd001.csv')
saved_train_units = sorted(saved_split[saved_split['split'] == 'train']['unit_number'].tolist())
saved_val_units   = sorted(saved_split[saved_split['split'] == 'val']['unit_number'].tolist())

# ── Step 3: Compare engine assignments ────────────────────────────────────
train_match = (regen_train_units == saved_train_units)
val_match   = (regen_val_units   == saved_val_units)

print(f'Regenerated train engines: {len(regen_train_units)}')
print(f'Saved train engines:       {len(saved_train_units)}')
print(f'Train assignment match:    {train_match}')
print(f'Val assignment match:      {val_match}')

if not train_match or not val_match:
    mismatched_train = set(regen_train_units) ^ set(saved_train_units)
    mismatched_val   = set(regen_val_units)   ^ set(saved_val_units)
    print(f'Mismatched train engines: {mismatched_train}')
    print(f'Mismatched val engines:   {mismatched_val}')
    raise RuntimeError(
        'GATE FAILED: Engine assignment mismatch. '
        'The split function does not reproduce the mid-semester split. '
        'Investigate before proceeding.'
    )

print('Engine assignment check: PASS')

=== SEED-42 REPRODUCTION GATE ===

Regenerated train engines: 80
Saved train engines:       80
Train assignment match:    True
Val assignment match:      True
Engine assignment check: PASS


In [9]:
# ── Step 4: Regenerate preprocessing for seed 42 ──────────────────────────
train_raw_42 = raw_train[raw_train['unit_number'].isin(regen_train_units)].copy()
val_raw_42   = raw_train[raw_train['unit_number'].isin(regen_val_units)].copy()

# Derived features
train_c_raw_42 = compute_derived_features(train_raw_42, SENSOR_COLS)
val_c_raw_42   = compute_derived_features(val_raw_42,   SENSOR_COLS)

# Feature lists
DERIVED_COLS = [
    c for c in train_c_raw_42.columns
    if c.endswith('_rmean') or c.endswith('_rstd') or c.endswith('_delta')
    or c == 'cycle_index'
]
FEATURE_SET_C = SENSOR_COLS + DERIVED_COLS

# Verify feature counts
assert len(SENSOR_COLS)  == 14, f'Expected 14 sensor cols, got {len(SENSOR_COLS)}'
assert len(DERIVED_COLS) == 43, f'Expected 43 derived cols, got {len(DERIVED_COLS)}'
assert 'cycle_index' in DERIVED_COLS
assert 'normalized_cycle_age' not in DERIVED_COLS

# Scale
train_b_42, val_b_42, train_c_42, val_c_42, scaler_b_42, scaler_c_42 = fit_and_apply_scalers(
    train_c_raw_42, val_c_raw_42, SENSOR_COLS, FEATURE_SET_C
)

print(f'Sensor cols (Feature Set B): {len(SENSOR_COLS)}')
print(f'Derived cols:                {len(DERIVED_COLS)}')
print(f'Feature Set C total:         {len(FEATURE_SET_C)}')
print(f'Train rows: {len(train_b_42)}  |  Val rows: {len(val_b_42)}')

Sensor cols (Feature Set B): 14
Derived cols:                43
Feature Set C total:         57
Train rows: 16340  |  Val rows: 4291


In [10]:
# ── Step 5: Build multi-view arrays and compare against saved NPZ ──────────
X_seq_42, X_der_42, y_42, meta_42 = create_multiview_windows(
    val_b_42, val_c_42, SENSOR_COLS, DERIVED_COLS, TARGET_COL
)
X_seq_train_42, X_der_train_42, y_train_42, train_meta_42 = create_multiview_windows(
    train_b_42, train_c_42, SENSOR_COLS, DERIVED_COLS, TARGET_COL
)

# Load saved NPZ and metadata
saved_npz       = np.load(f'{ARRAY_DIR}/val_multiview_fd001_window30.npz')
saved_meta      = pd.read_csv(f'{ARRAY_DIR}/val_multiview_meta_fd001_window30.csv')
saved_train_npz = np.load(f'{ARRAY_DIR}/train_multiview_fd001_window30.npz')
saved_train_meta= pd.read_csv(f'{ARRAY_DIR}/train_multiview_meta_fd001_window30.csv')

# ── Shape checks (val + train) ─────────────────────────────────────────────
shape_seq_match    = (X_seq_42.shape       == saved_npz['X_seq'].shape)
shape_der_match    = (X_der_42.shape       == saved_npz['X_derived'].shape)
shape_y_match      = (y_42.shape           == saved_npz['y'].shape)
shape_tr_seq_match = (X_seq_train_42.shape == saved_train_npz['X_seq'].shape)
shape_tr_der_match = (X_der_train_42.shape == saved_train_npz['X_derived'].shape)
shape_tr_y_match   = (y_train_42.shape     == saved_train_npz['y'].shape)

print(f'Val  X_seq  — regen: {X_seq_42.shape}        saved: {saved_npz["X_seq"].shape}        match: {shape_seq_match}')
print(f'Val  X_der  — regen: {X_der_42.shape}           saved: {saved_npz["X_derived"].shape}           match: {shape_der_match}')
print(f'Val  y      — regen: {y_42.shape}          saved: {saved_npz["y"].shape}          match: {shape_y_match}')
print(f'Train X_seq — regen: {X_seq_train_42.shape}      saved: {saved_train_npz["X_seq"].shape}      match: {shape_tr_seq_match}')
print(f'Train X_der — regen: {X_der_train_42.shape}        saved: {saved_train_npz["X_derived"].shape}        match: {shape_tr_der_match}')
print(f'Train y     — regen: {y_train_42.shape}      saved: {saved_train_npz["y"].shape}      match: {shape_tr_y_match}')

if not all([shape_seq_match, shape_der_match, shape_y_match,
            shape_tr_seq_match, shape_tr_der_match, shape_tr_y_match]):
    raise RuntimeError(
        'GATE FAILED: Array shape mismatch between regenerated and saved NPZ. '
        'Preprocessing functions may differ from NB03/NB06. Investigate.'
    )
print('Array shape check: PASS')

# ── Numerical comparison — validation arrays ───────────────────────────────
seq_close = np.allclose(X_seq_42,    saved_npz['X_seq'],     atol=1e-5, rtol=1e-5)
der_close = np.allclose(X_der_42,    saved_npz['X_derived'], atol=1e-5, rtol=1e-5)
y_close   = np.allclose(y_42,        saved_npz['y'],         atol=1e-5, rtol=1e-5)

# ── Numerical comparison — training arrays ─────────────────────────────────
tr_seq_close = np.allclose(X_seq_train_42, saved_train_npz['X_seq'],     atol=1e-5, rtol=1e-5)
tr_der_close = np.allclose(X_der_train_42, saved_train_npz['X_derived'], atol=1e-5, rtol=1e-5)
tr_y_close   = np.allclose(y_train_42,     saved_train_npz['y'],         atol=1e-5, rtol=1e-5)

print(f'Val  X_seq  numerical match (allclose): {seq_close}')
print(f'Val  X_der  numerical match (allclose): {der_close}')
print(f'Val  y      numerical match (allclose): {y_close}')
print(f'Train X_seq numerical match (allclose): {tr_seq_close}')
print(f'Train X_der numerical match (allclose): {tr_der_close}')
print(f'Train y     numerical match (allclose): {tr_y_close}')

for label, arr_close, arr_regen, arr_saved in [
    ('Val X_seq',   seq_close,    X_seq_42,    saved_npz['X_seq']),
    ('Val X_der',   der_close,    X_der_42,    saved_npz['X_derived']),
    ('Train X_seq', tr_seq_close, X_seq_train_42, saved_train_npz['X_seq']),
    ('Train X_der', tr_der_close, X_der_train_42, saved_train_npz['X_derived']),
]:
    if not arr_close:
        max_diff = float(np.abs(arr_regen - arr_saved).max())
        print(f'  Max absolute difference in {label}: {max_diff:.6f}')

arrays_all_close = all([seq_close, der_close, y_close, tr_seq_close, tr_der_close, tr_y_close])

# ── Metadata exact comparison — validation ─────────────────────────────────
META_COMPARE_COLS = ['unit_number', 'time_in_cycles', 'RUL', 'RUL_capped']
meta_rows_match  = (len(meta_42) == len(saved_meta))
meta_units_match = (sorted(meta_42['unit_number'].unique().tolist()) ==
                    sorted(saved_meta['unit_number'].unique().tolist()))
print(f'Val meta rows  — regen: {len(meta_42)}   saved: {len(saved_meta)}   match: {meta_rows_match}')
print(f'Val meta units match: {meta_units_match}')

meta_exact = False
if meta_rows_match:
    try:
        pd.testing.assert_frame_equal(
            meta_42[META_COMPARE_COLS].reset_index(drop=True),
            saved_meta[META_COMPARE_COLS].reset_index(drop=True),
            check_dtype=False, atol=1e-6, rtol=1e-6,
        )
        meta_exact = True
        print('Val metadata exact comparison (unit/cycle/RUL/RUL_capped): PASS')
    except AssertionError as e:
        print(f'Val metadata exact comparison: FAIL — {e}')
else:
    print('Val metadata exact comparison: SKIPPED (row count mismatch)')

# ── Metadata check — training ──────────────────────────────────────────────
train_meta_rows_match = (len(train_meta_42) == len(saved_train_meta))
train_meta_exact = False
if train_meta_rows_match:
    try:
        pd.testing.assert_frame_equal(
            train_meta_42[META_COMPARE_COLS].reset_index(drop=True),
            saved_train_meta[META_COMPARE_COLS].reset_index(drop=True),
            check_dtype=False, atol=1e-6, rtol=1e-6,
        )
        train_meta_exact = True
        print('Train metadata exact comparison: PASS')
    except AssertionError as e:
        print(f'Train metadata exact comparison: FAIL — {e}')
else:
    print(f'Train metadata row count mismatch: regen {len(train_meta_42)}  saved {len(saved_train_meta)}')

Val  X_seq  — regen: (3711, 30, 14)        saved: (3711, 30, 14)        match: True
Val  X_der  — regen: (3711, 43)           saved: (3711, 43)           match: True
Val  y      — regen: (3711,)          saved: (3711,)          match: True
Train X_seq — regen: (14020, 30, 14)      saved: (14020, 30, 14)      match: True
Train X_der — regen: (14020, 43)        saved: (14020, 43)        match: True
Train y     — regen: (14020,)      saved: (14020,)      match: True
Array shape check: PASS
Val  X_seq  numerical match (allclose): True
Val  X_der  numerical match (allclose): True
Val  y      numerical match (allclose): True
Train X_seq numerical match (allclose): True
Train X_der numerical match (allclose): True
Train y     numerical match (allclose): True
Val meta rows  — regen: 3711   saved: 3711   match: True
Val meta units match: True
Val metadata exact comparison (unit/cycle/RUL/RUL_capped): PASS
Train metadata exact comparison: PASS


In [11]:
# ── Step 6: Recalculate metrics from saved prediction CSVs ─────────────────
pred_fusion  = pd.read_csv(f'{BASE}/reports/predictions/val_predictions_MultiViewGRUFusion_window30_fd001.csv')
pred_gru     = pd.read_csv(f'{BASE}/reports/predictions/val_predictions_GRU_B_window30_fd001.csv')
pred_derived = pd.read_csv(f'{BASE}/reports/predictions/val_predictions_DerivedOnlyMLP_window30_fd001.csv')
pred_xgb_all = pd.read_csv(f'{BASE}/reports/predictions/val_predictions_XGBoost_C_fd001.csv')

# Verify no duplicate keys in prediction files
for name, df in [('Fusion', pred_fusion), ('GRU', pred_gru), ('DerivedOnlyMLP', pred_derived)]:
    dupes = df.duplicated(subset=['unit_number', 'time_in_cycles']).sum()
    assert dupes == 0, f'{name} predictions contain {dupes} duplicate (unit, cycle) keys'
dupes_xgb = pred_xgb_all.duplicated(subset=['unit_number', 'time_in_cycles']).sum()
assert dupes_xgb == 0, f'XGBoost predictions contain {dupes_xgb} duplicate keys'
print('No duplicate prediction keys in any file.')

# Confirm all windowed prediction files share identical (unit, cycle) keys
pd.testing.assert_frame_equal(
    pred_fusion[['unit_number', 'time_in_cycles']].reset_index(drop=True),
    pred_gru[['unit_number', 'time_in_cycles']].reset_index(drop=True),
    check_dtype=False,
)
pd.testing.assert_frame_equal(
    pred_fusion[['unit_number', 'time_in_cycles']].reset_index(drop=True),
    pred_derived[['unit_number', 'time_in_cycles']].reset_index(drop=True),
    check_dtype=False,
)
print('Fusion, GRU, DerivedOnlyMLP share identical (unit, cycle) keys.')

# XGBoost: align to the regenerated val metadata (not only to fusion prediction keys)
pred_xgb_aligned = pred_xgb_all.merge(
    meta_42[['unit_number', 'time_in_cycles']],
    on=['unit_number', 'time_in_cycles'],
    how='inner',
    validate='1:1',
)

print(f'XGBoost all-val rows:        {len(pred_xgb_all)}')
print(f'XGBoost window-aligned rows: {len(pred_xgb_aligned)}')
assert len(pred_xgb_aligned) == 3711, \
    f'Expected 3711 window-aligned XGBoost rows, got {len(pred_xgb_aligned)}'

# Recalculate all three metrics — full precision
m_fusion  = evaluate_predictions(pred_fusion['RUL_capped'].values,      pred_fusion['prediction'].values)
m_gru     = evaluate_predictions(pred_gru['RUL_capped'].values,         pred_gru['prediction'].values)
m_derived = evaluate_predictions(pred_derived['RUL_capped'].values,     pred_derived['prediction'].values)
m_xgb     = evaluate_predictions(pred_xgb_aligned['RUL_capped'].values, pred_xgb_aligned['prediction'].values)

recalc = {
    'MultiViewGRUFusion': m_fusion,
    'GRU':                m_gru,
    'DerivedOnlyMLP':     m_derived,
    'XGBoost':            m_xgb,
}

print()
print('Recalculated metrics from saved prediction CSVs:')
print(f'  {"Model":<25}  {"RMSE":>8}  {"ref RMSE":>8}  {"diff":>7}  {"MAE":>8}  {"ref MAE":>8}  {"R²":>7}  {"ref R²":>7}')
for model, m in recalc.items():
    ref  = REF_METRICS[model]
    diff = abs(m['rmse'] - ref['rmse'])
    status = 'PASS' if diff <= METRIC_TOLERANCE else 'FAIL'
    print(f'  {model:<25}  {m["rmse"]:8.4f}  {ref["rmse"]:8.4f}  {diff:7.4f}  '
          f'{m["mae"]:8.4f}  {ref["mae"]:8.4f}  {m["r2"]:7.4f}  {ref["r2"]:7.4f}  [{status}]')

# MAE and R² checked to four decimal places (metrics reproduced from stored files)
for model, m in recalc.items():
    ref = REF_METRICS[model]
    assert round(m['mae'], 4) == ref['mae'], \
        f'{model} MAE mismatch: recalc {round(m["mae"],4)} vs ref {ref["mae"]}'
    assert round(m['r2'], 4) == ref['r2'], \
        f'{model} R² mismatch: recalc {round(m["r2"],4)} vs ref {ref["r2"]}'
print('MAE and R² agreement to four decimal places: PASS')

No duplicate prediction keys in any file.
Fusion, GRU, DerivedOnlyMLP share identical (unit, cycle) keys.
XGBoost all-val rows:        4291
XGBoost window-aligned rows: 3711

Recalculated metrics from saved prediction CSVs:
  Model                          RMSE  ref RMSE     diff       MAE   ref MAE       R²   ref R²
  MultiViewGRUFusion          12.0657   12.0657   0.0000    8.9406    8.9406   0.9168   0.9168  [PASS]
  GRU                         13.1605   13.1605   0.0000    9.7182    9.7182   0.9010   0.9010  [PASS]
  DerivedOnlyMLP              13.1451   13.1451   0.0000    9.4204    9.4204   0.9013   0.9013  [PASS]
  XGBoost                     12.4894   12.4894   0.0000    9.2675    9.2675   0.9109   0.9109  [PASS]
MAE and R² agreement to four decimal places: PASS


In [12]:
# ── Step 7: Run leakage assertions for seed-42 ────────────────────────────
print('Running leakage assertions for seed-42 split...')
run_leakage_assertions(
    regen_train_units, regen_val_units,
    train_c_42, val_c_42,
    X_seq_train_42, X_seq_42,
    X_der_train_42, X_der_42,
    y_train_42, y_42,
    train_meta_42, meta_42,
    SENSOR_COLS, DERIVED_COLS,
)
print()

# ── Step 8: Build and print gate summary table ────────────────────────────
gate_rows = [
    ('Training engines',            N_TRAIN_ENG,  len(regen_train_units),
     'PASS' if len(regen_train_units) == N_TRAIN_ENG else 'FAIL'),
    ('Validation engines',          N_VAL_ENG,    len(regen_val_units),
     'PASS' if len(regen_val_units) == N_VAL_ENG else 'FAIL'),
    ('Training windows',            14020,         len(y_train_42),
     'PASS' if len(y_train_42) == 14020 else 'FAIL'),
    ('Validation windows',          3711,          len(y_42),
     'PASS' if len(y_42) == 3711 else 'FAIL'),
    ('Seq shape (per sample)',       '(30, 14)',    str(X_seq_42.shape[1:]),
     'PASS' if X_seq_42.shape[1:] == (30, 14) else 'FAIL'),
    ('Derived feature count',        43,            X_der_42.shape[1],
     'PASS' if X_der_42.shape[1] == 43 else 'FAIL'),
    ('Engine assignment',            'match',       'match' if train_match and val_match else 'MISMATCH',
     'PASS' if train_match and val_match else 'FAIL'),
    ('Val arrays numerical match',   'allclose',    'allclose' if (seq_close and der_close and y_close) else 'DIFFERS',
     'PASS' if (seq_close and der_close and y_close) else 'FAIL'),
    ('Train arrays numerical match', 'allclose',    'allclose' if (tr_seq_close and tr_der_close and tr_y_close) else 'DIFFERS',
     'PASS' if (tr_seq_close and tr_der_close and tr_y_close) else 'FAIL'),
    ('Val metadata and order',       'exact match', 'exact match' if meta_exact else 'MISMATCH',
     'PASS' if meta_exact else 'FAIL'),
    ('Train metadata and order',     'exact match', 'exact match' if train_meta_exact else 'MISMATCH',
     'PASS' if train_meta_exact else 'FAIL'),
    ('Fusion RMSE',                  REF_METRICS['MultiViewGRUFusion']['rmse'], round(m_fusion['rmse'],  4),
     'PASS' if abs(m_fusion['rmse']  - REF_METRICS['MultiViewGRUFusion']['rmse']) <= METRIC_TOLERANCE else 'FAIL'),
    ('Fusion MAE',                   REF_METRICS['MultiViewGRUFusion']['mae'],  round(m_fusion['mae'],   4),
     'PASS' if round(m_fusion['mae'],  4) == REF_METRICS['MultiViewGRUFusion']['mae'] else 'FAIL'),
    ('Fusion R²',                    REF_METRICS['MultiViewGRUFusion']['r2'],   round(m_fusion['r2'],    4),
     'PASS' if round(m_fusion['r2'],   4) == REF_METRICS['MultiViewGRUFusion']['r2']  else 'FAIL'),
    ('XGBoost aligned RMSE',         REF_METRICS['XGBoost']['rmse'],            round(m_xgb['rmse'],     4),
     'PASS' if abs(m_xgb['rmse']    - REF_METRICS['XGBoost']['rmse']) <= METRIC_TOLERANCE else 'FAIL'),
    ('XGBoost aligned MAE',          REF_METRICS['XGBoost']['mae'],             round(m_xgb['mae'],      4),
     'PASS' if round(m_xgb['mae'],    4) == REF_METRICS['XGBoost']['mae'] else 'FAIL'),
    ('XGBoost aligned R²',           REF_METRICS['XGBoost']['r2'],              round(m_xgb['r2'],       4),
     'PASS' if round(m_xgb['r2'],     4) == REF_METRICS['XGBoost']['r2']  else 'FAIL'),
    ('DerivedOnlyMLP RMSE',          REF_METRICS['DerivedOnlyMLP']['rmse'],     round(m_derived['rmse'], 4),
     'PASS' if abs(m_derived['rmse'] - REF_METRICS['DerivedOnlyMLP']['rmse']) <= METRIC_TOLERANCE else 'FAIL'),
    ('DerivedOnlyMLP MAE',           REF_METRICS['DerivedOnlyMLP']['mae'],      round(m_derived['mae'],  4),
     'PASS' if round(m_derived['mae'], 4) == REF_METRICS['DerivedOnlyMLP']['mae'] else 'FAIL'),
    ('DerivedOnlyMLP R²',            REF_METRICS['DerivedOnlyMLP']['r2'],       round(m_derived['r2'],   4),
     'PASS' if round(m_derived['r2'],  4) == REF_METRICS['DerivedOnlyMLP']['r2']  else 'FAIL'),
    ('GRU RMSE',                     REF_METRICS['GRU']['rmse'],                round(m_gru['rmse'],     4),
     'PASS' if abs(m_gru['rmse']    - REF_METRICS['GRU']['rmse']) <= METRIC_TOLERANCE else 'FAIL'),
    ('GRU MAE',                      REF_METRICS['GRU']['mae'],                 round(m_gru['mae'],      4),
     'PASS' if round(m_gru['mae'],    4) == REF_METRICS['GRU']['mae'] else 'FAIL'),
    ('GRU R²',                       REF_METRICS['GRU']['r2'],                  round(m_gru['r2'],       4),
     'PASS' if round(m_gru['r2'],     4) == REF_METRICS['GRU']['r2']  else 'FAIL'),
]

gate_df = pd.DataFrame(gate_rows, columns=['Check', 'Expected', 'Reproduced', 'Status'])
print('\n=== SEED-42 REPRODUCTION GATE SUMMARY ===')
print(gate_df.to_string(index=False))

all_pass             = all(r == 'PASS' for r in gate_df['Status'])
passed_with_warnings = False   # no WARN status exists in this gate

print()
if all_pass:
    print('GATE RESULT: ALL CHECKS PASSED — seed 21 and seed 84 training may proceed.')
else:
    failed = gate_df[gate_df['Status'] == 'FAIL']['Check'].tolist()
    raise RuntimeError(
        f'GATE FAILED on: {failed}. '
        'Do not proceed to seed 21/84 training until failures are resolved.'
    )

Running leakage assertions for seed-42 split...
  All configured leakage-related checks passed.


=== SEED-42 REPRODUCTION GATE SUMMARY ===
                       Check    Expected  Reproduced Status
            Training engines          80          80   PASS
          Validation engines          20          20   PASS
            Training windows       14020       14020   PASS
          Validation windows        3711        3711   PASS
      Seq shape (per sample)    (30, 14)    (30, 14)   PASS
       Derived feature count          43          43   PASS
           Engine assignment       match       match   PASS
  Val arrays numerical match    allclose    allclose   PASS
Train arrays numerical match    allclose    allclose   PASS
      Val metadata and order exact match exact match   PASS
    Train metadata and order exact match exact match   PASS
                 Fusion RMSE     12.0657     12.0657   PASS
                  Fusion MAE      8.9406      8.9406   PASS
                   F

In [13]:
# ── Step 9: Save gate artefacts ────────────────────────────────────────────
gate_csv_path = f'{RV_DIR}/seed42_reproduction_gate.csv'
gate_df.to_csv(gate_csv_path, index=False)
print(f'Gate table saved: {gate_csv_path}')

# Save exact derived feature names now that they are confirmed
derived_feature_names_path = f'{RV_DIR}/derived_feature_names.json'
with open(derived_feature_names_path, 'w') as f:
    json.dump({'derived_cols': DERIVED_COLS, 'n_derived': len(DERIVED_COLS)}, f, indent=2)

gate_manifest = {
    'gate_run_at':              datetime.now(timezone.utc).isoformat(),
    'reference_split_seed':     REFERENCE_SPLIT_SEED,
    'model_seed':               MODEL_SEED,
    'metric_tolerance':         METRIC_TOLERANCE,
    'all_pass':                 bool(all_pass),
    'passed_with_warnings':     False,
    'train_assignment_match':   bool(train_match),
    'val_assignment_match':     bool(val_match),
    'val_arrays_all_close':     bool(seq_close and der_close and y_close),
    'train_arrays_all_close':   bool(tr_seq_close and tr_der_close and tr_y_close),
    'val_metadata_exact':       bool(meta_exact),
    'train_metadata_exact':     bool(train_meta_exact),
    'recalculated_metrics':     recalc,
    'reference_metrics':        REF_METRICS,
    'n_train_windows':          int(len(y_train_42)),
    'n_val_windows':            int(len(y_42)),
    'n_train_engines':          int(len(regen_train_units)),
    'n_val_engines':            int(len(regen_val_units)),
    'derived_col_count':        int(len(DERIVED_COLS)),
    'sensor_col_count':         int(len(SENSOR_COLS)),
    'gate_checks':              gate_df.to_dict(orient='records'),
}

manifest_path = f'{RV_DIR}/seed42_reproduction_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(gate_manifest, f, indent=2)
print(f'Gate manifest saved: {manifest_path}')
print(f'Derived feature names saved: {derived_feature_names_path}')

print()
print('=== SECTION 7 COMPLETE ===')
print('Review gate output above. Proceed to Section 8 (seed 21/84 training) only if gate passed.')

Gate table saved: /content/drive/MyDrive/Dissertation/Project/dissertation-rul-xai/reports/final_validation/seed42_reproduction_gate.csv
Gate manifest saved: /content/drive/MyDrive/Dissertation/Project/dissertation-rul-xai/reports/final_validation/seed42_reproduction_manifest.json
Derived feature names saved: /content/drive/MyDrive/Dissertation/Project/dissertation-rul-xai/reports/final_validation/derived_feature_names.json

=== SECTION 7 COMPLETE ===
Review gate output above. Proceed to Section 8 (seed 21/84 training) only if gate passed.


## Section 7.1 — Gate Artefacts

The following files are written by the seed-42 reproduction gate. Per-split training outputs, consolidation tables, and ablation results are listed in the complete artefact listing at the end of the notebook.

In [14]:
import os

print('Generated artefacts:\n')

print('Reports — final_validation:')
if os.path.isdir(RV_DIR):
    for f in sorted(os.listdir(RV_DIR)):
        if not f.startswith('.'):
            size_kb = os.path.getsize(f'{RV_DIR}/{f}') / 1024
            print(f'  {f} ({size_kb:.1f} KB)')
else:
    print('  (directory not yet created)')

print()
print('Note: model and per-split artefacts will appear here after seed 21/84 training.')
print('Gate artefacts saved above are the only outputs from the current execution.')

Generated artefacts:

Reports — final_validation:
  derived_feature_names.json (1.5 KB)
  experiment_environment.json (0.4 KB)
  frozen_experiment_config.json (2.4 KB)
  seed42_reproduction_gate.csv (0.9 KB)
  seed42_reproduction_manifest.json (4.3 KB)

Note: model and per-split artefacts will appear here after seed 21/84 training.
Gate artefacts saved above are the only outputs from the current execution.


## Section 8 — Repeated Validation: Seeds 21 and 84

The seed-42 reproduction gate has passed. I now train the same four frozen model configurations on two additional engine-level split configurations — split seeds 21 and 84 — keeping MODEL_SEED fixed at 42 throughout.

The `run_split_experiment` function below implements the full pipeline for a single split seed: engine assignment, preprocessing, leakage assertions, model training, validation, and artefact saving. I run seed 21 first and verify all its outputs before running seed 84. No configuration changes are made between seeds.

All outputs are written under `data/processed/final_validation/split_seed_XX/`, `models/final_validation/split_seed_XX/`, and `reports/final_validation/split_seed_XX/`. No mid-semester artefact is overwritten.

**Reminder:** Before each neural model, the function calls `clear_session()`, `gc.collect()`, and `set_random_seed(MODEL_SEED)` independently — not once per split.

In [ ]:
def run_split_experiment(split_seed):
    """
    Full pipeline for one engine-level split configuration.
    Supports resumption: when OVERWRITE_EXISTING=False and all artefacts exist,
    reloads saved metrics and returns without retraining.
    Returns a dict of per-model metrics for consolidation.
    """
    print(f'
{"="*60}')
    print(f'SPLIT SEED {split_seed} — START')
    print(f'{"="*60}')

    # ── Directory setup ────────────────────────────────────────────────────
    fv_dir = f'{FV_BASE}/split_seed_{split_seed}'
    mv_dir = f'{MV_BASE}/split_seed_{split_seed}'
    rv_dir = f'{RV_DIR}/split_seed_{split_seed}'
    for d in [fv_dir, mv_dir, rv_dir]:
        os.makedirs(d, exist_ok=True)

    # ── Resume check ───────────────────────────────────────────────────────
    manifest_path = f'{rv_dir}/run_manifest.json'
    required_artefacts = [
        f'{fv_dir}/split_assignments.csv', f'{fv_dir}/feature_lists.json',
        f'{fv_dir}/scaler_b.joblib',       f'{fv_dir}/scaler_c.joblib',
        f'{mv_dir}/XGBoost_C.joblib',      f'{mv_dir}/GRU_B_window30.keras',
        f'{mv_dir}/DerivedOnlyMLP_window30.keras',
        f'{mv_dir}/MultiViewGRUFusion_window30.keras',
        f'{rv_dir}/model_metrics.csv',     f'{rv_dir}/per_engine_metrics.csv',
        f'{rv_dir}/predictions_XGBoost.csv',
        f'{rv_dir}/predictions_GRU.csv',
        f'{rv_dir}/predictions_DerivedOnlyMLP.csv',
        f'{rv_dir}/predictions_MultiViewGRUFusion.csv',
        f'{rv_dir}/training_history_GRU.csv',
        f'{rv_dir}/training_history_DerivedOnlyMLP.csv',
        f'{rv_dir}/training_history_MultiViewGRUFusion.csv',
        manifest_path,
    ]
    if not OVERWRITE_EXISTING and all(os.path.isfile(p) for p in required_artefacts):
        print(f'  All artefacts found and OVERWRITE_EXISTING=False — loading saved results.')
        with open(manifest_path) as f:
            saved = json.load(f)
        results = {}
        for model_name, m in saved['results'].items():
            results[model_name] = {
                'rmse':              m['rmse'],
                'mae':               m['mae'],
                'r2':                m['r2'],
                'best_epoch':        m.get('best_epoch'),
                'macro_engine_rmse': m.get('macro_engine_rmse'),
                'mean_error':        m.get('mean_error'),
            }
        print(f'  Loaded results for: {list(results.keys())}')
        return results

    results = {}

    # ── 1. Engine split ────────────────────────────────────────────────────
    train_units, val_units = make_engine_split(all_units, N_TRAIN_ENG, split_seed)
    split_df = pd.DataFrame(
        [{'unit_number': u, 'split': 'train'} for u in train_units] +
        [{'unit_number': u, 'split': 'val'}   for u in val_units]
    )
    split_df.to_csv(f'{fv_dir}/split_assignments.csv', index=False)
    print(f'  Train engines: {len(train_units)}  |  Val engines: {len(val_units)}')

    # ── 2. Preprocessing ───────────────────────────────────────────────────
    train_raw = raw_train[raw_train['unit_number'].isin(train_units)].copy()
    val_raw   = raw_train[raw_train['unit_number'].isin(val_units)].copy()

    train_c_raw = compute_derived_features(train_raw, SENSOR_COLS)
    val_c_raw   = compute_derived_features(val_raw,   SENSOR_COLS)

    derived_cols = [
        c for c in train_c_raw.columns
        if c.endswith('_rmean') or c.endswith('_rstd') or c.endswith('_delta')
        or c == 'cycle_index'
    ]
    feature_set_c = SENSOR_COLS + derived_cols
    assert len(derived_cols) == 43
    assert 'cycle_index' in derived_cols
    assert 'normalized_cycle_age' not in derived_cols

    train_b, val_b, train_c, val_c, scaler_b, scaler_c = fit_and_apply_scalers(
        train_c_raw, val_c_raw, SENSOR_COLS, feature_set_c
    )

    joblib.dump(scaler_b, f'{fv_dir}/scaler_b.joblib')
    joblib.dump(scaler_c, f'{fv_dir}/scaler_c.joblib')
    with open(f'{fv_dir}/feature_lists.json', 'w') as f:
        json.dump({'sensor_cols': SENSOR_COLS, 'derived_cols': derived_cols,
                   'feature_set_c': feature_set_c}, f, indent=2)

    # ── 3. Window arrays ───────────────────────────────────────────────────
    X_seq_tr, X_der_tr, y_tr, meta_tr = create_multiview_windows(
        train_b, train_c, SENSOR_COLS, derived_cols, TARGET_COL)
    X_seq_val, X_der_val, y_val, meta_val = create_multiview_windows(
        val_b, val_c, SENSOR_COLS, derived_cols, TARGET_COL)

    print(f'  Train windows: {len(y_tr)}  |  Val windows: {len(y_val)}')

    # ── 4. Leakage assertions ──────────────────────────────────────────────
    run_leakage_assertions(
        train_units, val_units, train_c, val_c,
        X_seq_tr, X_seq_val, X_der_tr, X_der_val,
        y_tr, y_val, meta_tr, meta_val,
        SENSOR_COLS, derived_cols,
    )

    # ── 5. XGBoost ────────────────────────────────────────────────────────
    print(f'
  [XGBoost] training...')
    xgb_model = XGBRegressor(**XGB_PARAMS)
    xgb_model.fit(train_c[feature_set_c], train_c[TARGET_COL])

    val_c_aligned = val_c.merge(
        meta_val[['unit_number', 'time_in_cycles']],
        on=['unit_number', 'time_in_cycles'], how='inner'
    )
    assert len(val_c_aligned) == len(meta_val),         f'val_c_aligned rows {len(val_c_aligned)} != meta_val {len(meta_val)}'
    pd.testing.assert_frame_equal(
        val_c_aligned[['unit_number', 'time_in_cycles']].reset_index(drop=True),
        meta_val[['unit_number', 'time_in_cycles']].reset_index(drop=True),
        check_dtype=False,
    )

    xgb_pred_aligned = np.clip(xgb_model.predict(val_c_aligned[feature_set_c]), 0, RUL_CAP)
    xgb_m = evaluate_predictions(val_c_aligned[TARGET_COL].values, xgb_pred_aligned)
    per_eng_xgb = compute_per_engine_metrics(
        val_c_aligned[TARGET_COL].values, xgb_pred_aligned, meta_val)

    joblib.dump(xgb_model, f'{mv_dir}/XGBoost_C.joblib')
    xgb_pred_df = val_c_aligned[['unit_number', 'time_in_cycles', TARGET_COL]].copy()
    xgb_pred_df['prediction'] = xgb_pred_aligned
    xgb_pred_df.to_csv(f'{rv_dir}/predictions_XGBoost.csv', index=False)

    results['XGBoost'] = {
        **xgb_m,
        'best_epoch':        None,
        'macro_engine_rmse': round(float(per_eng_xgb['engine_rmse'].mean()), 6),
        'mean_error':        round(float((xgb_pred_aligned - val_c_aligned[TARGET_COL].values).mean()), 6),
    }
    print(f'  [XGBoost] RMSE={xgb_m["rmse"]:.4f}  MAE={xgb_m["mae"]:.4f}  R2={xgb_m["r2"]:.4f}  '
          f'macro_eng_rmse={results["XGBoost"]["macro_engine_rmse"]:.4f}  '
          f'mean_err={results["XGBoost"]["mean_error"]:.4f}')

    # ── 6. GRU ────────────────────────────────────────────────────────────
    print(f'
  [GRU] training...')
    tf.keras.backend.clear_session(); gc.collect()
    tf.keras.utils.set_random_seed(MODEL_SEED)

    gru = build_gru(WINDOW_SIZE, len(SENSOR_COLS))
    gru_hist = gru.fit(
        X_seq_tr, y_tr,
        validation_data=(X_seq_val, y_val),
        epochs=GRU_MAX_EPOCHS, batch_size=GRU_BATCH,
        callbacks=get_gru_callbacks(), verbose=0,
    )
    gru_best_epoch = int(np.argmin(gru_hist.history['val_loss'])) + 1
    gru_pred = np.clip(gru.predict(X_seq_val, verbose=0).ravel(), 0, RUL_CAP)
    gru_m = evaluate_predictions(y_val, gru_pred)
    per_eng_gru = compute_per_engine_metrics(y_val, gru_pred, meta_val)

    gru.save(f'{mv_dir}/GRU_B_window30.keras')
    pd.DataFrame(gru_hist.history).to_csv(f'{rv_dir}/training_history_GRU.csv', index=False)
    pred_df = meta_val[['unit_number', 'time_in_cycles', 'RUL_capped']].copy()
    pred_df['prediction'] = gru_pred
    pred_df.to_csv(f'{rv_dir}/predictions_GRU.csv', index=False)

    results['GRU'] = {
        **gru_m,
        'best_epoch':        gru_best_epoch,
        'macro_engine_rmse': round(float(per_eng_gru['engine_rmse'].mean()), 6),
        'mean_error':        round(float((gru_pred - y_val).mean()), 6),
    }
    print(f'  [GRU] best_epoch={gru_best_epoch}  RMSE={gru_m["rmse"]:.4f}  '
          f'MAE={gru_m["mae"]:.4f}  R2={gru_m["r2"]:.4f}')

    # ── 7. DerivedOnlyMLP ─────────────────────────────────────────────────
    print(f'
  [DerivedOnlyMLP] training...')
    tf.keras.backend.clear_session(); gc.collect()
    tf.keras.utils.set_random_seed(MODEL_SEED)

    mlp = build_derived_mlp(len(derived_cols))
    mlp_hist = mlp.fit(
        X_der_tr, y_tr,
        validation_data=(X_der_val, y_val),
        epochs=MLP_MAX_EPOCHS, batch_size=MLP_BATCH,
        callbacks=get_mlp_callbacks(), verbose=0,
    )
    mlp_best_epoch = int(np.argmin(mlp_hist.history['val_loss'])) + 1
    mlp_pred = np.clip(mlp.predict(X_der_val, verbose=0).ravel(), 0, RUL_CAP)
    mlp_m = evaluate_predictions(y_val, mlp_pred)
    per_eng_mlp = compute_per_engine_metrics(y_val, mlp_pred, meta_val)

    mlp.save(f'{mv_dir}/DerivedOnlyMLP_window30.keras')
    pd.DataFrame(mlp_hist.history).to_csv(f'{rv_dir}/training_history_DerivedOnlyMLP.csv', index=False)
    pred_df = meta_val[['unit_number', 'time_in_cycles', 'RUL_capped']].copy()
    pred_df['prediction'] = mlp_pred
    pred_df.to_csv(f'{rv_dir}/predictions_DerivedOnlyMLP.csv', index=False)

    results['DerivedOnlyMLP'] = {
        **mlp_m,
        'best_epoch':        mlp_best_epoch,
        'macro_engine_rmse': round(float(per_eng_mlp['engine_rmse'].mean()), 6),
        'mean_error':        round(float((mlp_pred - y_val).mean()), 6),
    }
    print(f'  [DerivedOnlyMLP] best_epoch={mlp_best_epoch}  RMSE={mlp_m["rmse"]:.4f}  '
          f'MAE={mlp_m["mae"]:.4f}  R2={mlp_m["r2"]:.4f}')

    # ── 8. MultiViewGRUFusion ─────────────────────────────────────────────
    print(f'
  [MultiViewGRUFusion] training...')
    tf.keras.backend.clear_session(); gc.collect()
    tf.keras.utils.set_random_seed(MODEL_SEED)

    fusion = build_multiview_gru(WINDOW_SIZE, len(SENSOR_COLS), len(derived_cols))
    fusion_hist = fusion.fit(
        [X_seq_tr, X_der_tr], y_tr,
        validation_data=([X_seq_val, X_der_val], y_val),
        epochs=MLP_MAX_EPOCHS, batch_size=MLP_BATCH,
        callbacks=get_mlp_callbacks(), verbose=0,
    )
    fusion_best_epoch = int(np.argmin(fusion_hist.history['val_loss'])) + 1
    fusion_pred = np.clip(fusion.predict([X_seq_val, X_der_val], verbose=0).ravel(), 0, RUL_CAP)
    fusion_m = evaluate_predictions(y_val, fusion_pred)
    per_eng_fusion = compute_per_engine_metrics(y_val, fusion_pred, meta_val)

    fusion.save(f'{mv_dir}/MultiViewGRUFusion_window30.keras')
    pd.DataFrame(fusion_hist.history).to_csv(f'{rv_dir}/training_history_MultiViewGRUFusion.csv', index=False)
    pred_df = meta_val[['unit_number', 'time_in_cycles', 'RUL_capped']].copy()
    pred_df['prediction'] = fusion_pred
    pred_df.to_csv(f'{rv_dir}/predictions_MultiViewGRUFusion.csv', index=False)

    results['MultiViewGRUFusion'] = {
        **fusion_m,
        'best_epoch':        fusion_best_epoch,
        'macro_engine_rmse': round(float(per_eng_fusion['engine_rmse'].mean()), 6),
        'mean_error':        round(float((fusion_pred - y_val).mean()), 6),
    }
    print(f'  [MultiViewGRUFusion] best_epoch={fusion_best_epoch}  RMSE={fusion_m["rmse"]:.4f}  '
          f'MAE={fusion_m["mae"]:.4f}  R2={fusion_m["r2"]:.4f}')

    # ── 9. Per-engine metrics (all models combined) ────────────────────────
    per_eng_rows = []
    for model_name, pred_arr, yt, mt in [
        ('XGBoost',            xgb_pred_aligned,  val_c_aligned[TARGET_COL].values, meta_val),
        ('GRU',                gru_pred,           y_val,                            meta_val),
        ('DerivedOnlyMLP',     mlp_pred,           y_val,                            meta_val),
        ('MultiViewGRUFusion', fusion_pred,        y_val,                            meta_val),
    ]:
        per_e = compute_per_engine_metrics(yt, pred_arr, mt)
        per_e.insert(0, 'model', model_name)
        per_e.insert(1, 'split_seed', split_seed)
        per_eng_rows.append(per_e)
    per_eng_df = pd.concat(per_eng_rows, ignore_index=True)
    per_eng_df.to_csv(f'{rv_dir}/per_engine_metrics.csv', index=False)

    # ── 10. Split model_metrics summary ───────────────────────────────────
    metric_rows = []
    for model_name, m in results.items():
        metric_rows.append({
            'split_seed':        split_seed,
            'model_seed':        MODEL_SEED,
            'model':             model_name,
            'validation_rmse':   round(m['rmse'], 6),
            'validation_mae':    round(m['mae'],  6),
            'validation_r2':     round(m['r2'],   6),
            'macro_engine_rmse': m['macro_engine_rmse'],
            'best_epoch':        m['best_epoch'],
            'mean_error':        m['mean_error'],
        })
    pd.DataFrame(metric_rows).to_csv(f'{rv_dir}/model_metrics.csv', index=False)

    # ── 11. Run manifest ───────────────────────────────────────────────────
    manifest = {
        'split_seed':        split_seed,
        'model_seed':        MODEL_SEED,
        'completed_at':      datetime.now(timezone.utc).isoformat(),
        'n_train_engines':   len(train_units),
        'n_val_engines':     len(val_units),
        'n_train_windows':   int(len(y_tr)),
        'n_val_windows':     int(len(y_val)),
        'derived_col_count': len(derived_cols),
        'results':           {k: {kk: round(vv, 6) if isinstance(vv, float) else vv
                                  for kk, vv in v.items()} for k, v in results.items()},
    }
    with open(f'{rv_dir}/run_manifest.json', 'w') as f:
        json.dump(manifest, f, indent=2)

    print(f'
  SPLIT SEED {split_seed} — COMPLETE')
    for model_name, m in results.items():
        print(f'    {model_name:<25}  RMSE={m["rmse"]:.4f}  MAE={m["mae"]:.4f}  '
              f'R2={m["r2"]:.4f}  macro_eng={m["macro_engine_rmse"]}  '
              f'mean_err={m["mean_error"]}')

    return results


print('run_split_experiment() defined.')

### Split Seed 21

I run the full pipeline for split seed 21 — engine assignment, preprocessing, leakage assertions, XGBoost, GRU, DerivedOnlyMLP, and MultiViewGRUFusion — then verify that all expected artefacts are present before proceeding to seed 84.

In [16]:
results_21 = run_split_experiment(21)


SPLIT SEED 21 — START
  Train engines: 80  |  Val engines: 20
  Train windows: 14348  |  Val windows: 3383
  All configured leakage-related checks passed.

  [XGBoost] training...
  [XGBoost] RMSE=14.1129  MAE=9.9489  R²=0.8850

  [GRU] training...

Epoch 17: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 22: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
Epoch 22: early stopping
Restoring model weights from the end of the best epoch: 12.
  [GRU] best_epoch=12  RMSE=14.5044  MAE=11.1430  R²=0.8785

  [DerivedOnlyMLP] training...

Epoch 59: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
Restoring model weights from the end of the best epoch: 55.
  [DerivedOnlyMLP] best_epoch=55  RMSE=14.0158  MAE=9.9778  R²=0.8865

  [MultiViewGRUFusion] training...

Epoch 16: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 20: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
Epoch 20: early s

In [17]:
# ── Seed-21 artefact verification ─────────────────────────────────────────
print('Verifying seed-21 artefacts...\n')

SEED21_DATA   = f'{FV_BASE}/split_seed_21'
SEED21_MODELS = f'{MV_BASE}/split_seed_21'
SEED21_RPTS   = f'{RV_DIR}/split_seed_21'

expected_files = {
    'data': [
        f'{SEED21_DATA}/split_assignments.csv',
        f'{SEED21_DATA}/feature_lists.json',
        f'{SEED21_DATA}/scaler_b.joblib',
        f'{SEED21_DATA}/scaler_c.joblib',
    ],
    'models': [
        f'{SEED21_MODELS}/XGBoost_C.joblib',
        f'{SEED21_MODELS}/GRU_B_window30.keras',
        f'{SEED21_MODELS}/DerivedOnlyMLP_window30.keras',
        f'{SEED21_MODELS}/MultiViewGRUFusion_window30.keras',
    ],
    'reports': [
        f'{SEED21_RPTS}/model_metrics.csv',
        f'{SEED21_RPTS}/per_engine_metrics.csv',
        f'{SEED21_RPTS}/predictions_XGBoost.csv',
        f'{SEED21_RPTS}/predictions_GRU.csv',
        f'{SEED21_RPTS}/predictions_DerivedOnlyMLP.csv',
        f'{SEED21_RPTS}/predictions_MultiViewGRUFusion.csv',
        f'{SEED21_RPTS}/training_history_GRU.csv',
        f'{SEED21_RPTS}/training_history_DerivedOnlyMLP.csv',
        f'{SEED21_RPTS}/training_history_MultiViewGRUFusion.csv',
        f'{SEED21_RPTS}/run_manifest.json',
    ],
}

all_present = True
for category, paths in expected_files.items():
    print(f'{category}:')
    for p in paths:
        exists = os.path.isfile(p)
        size_kb = os.path.getsize(p) / 1024 if exists else 0
        status = f'OK ({size_kb:.1f} KB)' if exists else 'MISSING'
        print(f'  {"OK" if exists else "!!"} {os.path.basename(p)} — {status}')
        if not exists:
            all_present = False

assert all_present, 'One or more seed-21 artefacts are missing. Do not proceed to seed 84.'

# Print manifest
with open(f'{SEED21_RPTS}/run_manifest.json') as f:
    m21 = json.load(f)
print(f'\nSeed-21 manifest:')
print(f'  completed_at:    {m21["completed_at"]}')
print(f'  n_train_engines: {m21["n_train_engines"]}')
print(f'  n_val_engines:   {m21["n_val_engines"]}')
print(f'  n_train_windows: {m21["n_train_windows"]}')
print(f'  n_val_windows:   {m21["n_val_windows"]}')
print(f'  Results:')
for model_name, m in m21['results'].items():
    print(f'    {model_name:<25}  RMSE={m["rmse"]:.4f}  MAE={m["mae"]:.4f}  R²={m["r2"]:.4f}')

print('\nSeed-21 artefact check: PASS — proceed to seed 84.')

Verifying seed-21 artefacts...

data:
  OK split_assignments.csv — OK (0.8 KB)
  OK feature_lists.json — OK (3.7 KB)
  OK scaler_b.joblib — OK (1.5 KB)
  OK scaler_c.joblib — OK (3.7 KB)
models:
  OK XGBoost_C.joblib — OK (514.8 KB)
  OK GRU_B_window30.keras — OK (349.0 KB)
  OK DerivedOnlyMLP_window30.keras — OK (87.4 KB)
  OK MultiViewGRUFusion_window30.keras — OK (393.0 KB)
reports:
  OK model_metrics.csv — OK (0.4 KB)
  OK per_engine_metrics.csv — OK (6.1 KB)
  OK predictions_XGBoost.csv — OK (70.6 KB)
  OK predictions_GRU.csv — OK (71.3 KB)
  OK predictions_DerivedOnlyMLP.csv — OK (69.8 KB)
  OK predictions_MultiViewGRUFusion.csv — OK (70.7 KB)
  OK training_history_GRU.csv — OK (2.1 KB)
  OK training_history_DerivedOnlyMLP.csv — OK (5.7 KB)
  OK training_history_MultiViewGRUFusion.csv — OK (1.9 KB)
  OK run_manifest.json — OK (1.0 KB)

Seed-21 manifest:
  completed_at:    2026-07-27T00:40:40.465061+00:00
  n_train_engines: 80
  n_val_engines:   20
  n_train_windows: 14348
  n_val

### Split Seed 84

Seed-21 artefacts verified. I now run the identical pipeline for split seed 84 with no configuration changes. MODEL_SEED remains 42.

In [18]:
results_84 = run_split_experiment(84)


SPLIT SEED 84 — START
  Train engines: 80  |  Val engines: 20
  Train windows: 13983  |  Val windows: 3748
  All configured leakage-related checks passed.

  [XGBoost] training...
  [XGBoost] RMSE=16.6421  MAE=12.2032  R²=0.8421

  [GRU] training...

Epoch 22: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
Restoring model weights from the end of the best epoch: 26.
  [GRU] best_epoch=26  RMSE=14.8684  MAE=11.5329  R²=0.8740

  [DerivedOnlyMLP] training...
Restoring model weights from the end of the best epoch: 59.
  [DerivedOnlyMLP] best_epoch=59  RMSE=15.2359  MAE=11.0725  R²=0.8677

  [MultiViewGRUFusion] training...

Epoch 16: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 20: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
Epoch 20: early stopping
Restoring model weights from the end of the best epoch: 12.
  [MultiViewGRUFusion] best_epoch=12  RMSE=15.7599  MAE=12.1711  R²=0.8584

  SPLIT SEED 84 — COMPLETE
  Result

In [19]:
# ── Seed-84 artefact verification ─────────────────────────────────────────
print('Verifying seed-84 artefacts...\n')

SEED84_DATA   = f'{FV_BASE}/split_seed_84'
SEED84_MODELS = f'{MV_BASE}/split_seed_84'
SEED84_RPTS   = f'{RV_DIR}/split_seed_84'

expected_files_84 = {
    'data': [
        f'{SEED84_DATA}/split_assignments.csv',
        f'{SEED84_DATA}/feature_lists.json',
        f'{SEED84_DATA}/scaler_b.joblib',
        f'{SEED84_DATA}/scaler_c.joblib',
    ],
    'models': [
        f'{SEED84_MODELS}/XGBoost_C.joblib',
        f'{SEED84_MODELS}/GRU_B_window30.keras',
        f'{SEED84_MODELS}/DerivedOnlyMLP_window30.keras',
        f'{SEED84_MODELS}/MultiViewGRUFusion_window30.keras',
    ],
    'reports': [
        f'{SEED84_RPTS}/model_metrics.csv',
        f'{SEED84_RPTS}/per_engine_metrics.csv',
        f'{SEED84_RPTS}/predictions_XGBoost.csv',
        f'{SEED84_RPTS}/predictions_GRU.csv',
        f'{SEED84_RPTS}/predictions_DerivedOnlyMLP.csv',
        f'{SEED84_RPTS}/predictions_MultiViewGRUFusion.csv',
        f'{SEED84_RPTS}/training_history_GRU.csv',
        f'{SEED84_RPTS}/training_history_DerivedOnlyMLP.csv',
        f'{SEED84_RPTS}/training_history_MultiViewGRUFusion.csv',
        f'{SEED84_RPTS}/run_manifest.json',
    ],
}

all_present_84 = True
for category, paths in expected_files_84.items():
    print(f'{category}:')
    for p in paths:
        exists = os.path.isfile(p)
        size_kb = os.path.getsize(p) / 1024 if exists else 0
        status = f'OK ({size_kb:.1f} KB)' if exists else 'MISSING'
        print(f'  {"OK" if exists else "!!"} {os.path.basename(p)} — {status}')
        if not exists:
            all_present_84 = False

assert all_present_84, 'One or more seed-84 artefacts are missing. Do not proceed to consolidation.'

with open(f'{SEED84_RPTS}/run_manifest.json') as f:
    m84 = json.load(f)
print(f'\nSeed-84 manifest:')
print(f'  completed_at:    {m84["completed_at"]}')
print(f'  n_train_engines: {m84["n_train_engines"]}')
print(f'  n_val_engines:   {m84["n_val_engines"]}')
print(f'  n_train_windows: {m84["n_train_windows"]}')
print(f'  n_val_windows:   {m84["n_val_windows"]}')
print(f'  Results:')
for model_name, m in m84['results'].items():
    print(f'    {model_name:<25}  RMSE={m["rmse"]:.4f}  MAE={m["mae"]:.4f}  R²={m["r2"]:.4f}')

print('\nSeed-84 artefact check: PASS — proceed to consolidation.')

Verifying seed-84 artefacts...

data:
  OK split_assignments.csv — OK (0.8 KB)
  OK feature_lists.json — OK (3.7 KB)
  OK scaler_b.joblib — OK (1.5 KB)
  OK scaler_c.joblib — OK (3.7 KB)
models:
  OK XGBoost_C.joblib — OK (514.3 KB)
  OK GRU_B_window30.keras — OK (349.0 KB)
  OK DerivedOnlyMLP_window30.keras — OK (87.4 KB)
  OK MultiViewGRUFusion_window30.keras — OK (393.0 KB)
reports:
  OK model_metrics.csv — OK (0.4 KB)
  OK per_engine_metrics.csv — OK (6.1 KB)
  OK predictions_XGBoost.csv — OK (78.9 KB)
  OK predictions_GRU.csv — OK (80.0 KB)
  OK predictions_DerivedOnlyMLP.csv — OK (78.4 KB)
  OK predictions_MultiViewGRUFusion.csv — OK (79.3 KB)
  OK training_history_GRU.csv — OK (2.8 KB)
  OK training_history_DerivedOnlyMLP.csv — OK (5.6 KB)
  OK training_history_MultiViewGRUFusion.csv — OK (1.9 KB)
  OK run_manifest.json — OK (1.0 KB)

Seed-84 manifest:
  completed_at:    2026-07-27T00:42:53.868876+00:00
  n_train_engines: 80
  n_val_engines:   20
  n_train_windows: 13983
  n_val

### Artefact Repair: Seeds 21 and 84 XGBoost Fields

The first run of seeds 21 and 84 saved  for XGBoost  and  due to an incomplete runner. This cell recomputes those fields from the saved prediction CSVs — no retraining is performed — and patches the  and  for both splits. An audit log is saved to confirm the repair.

In [ ]:
# ── Artefact repair: XGBoost macro_engine_rmse and mean_error ──────────────
# Seeds 21 and 84 were saved with None for these fields.
# Recomputed from saved prediction CSVs — no model retraining.
print('Repairing XGBoost artefacts for seeds 21 and 84...
')

repair_log = {
    'repaired_at': datetime.now(timezone.utc).isoformat(),
    'seeds_repaired': [],
    'fields_recalculated': ['macro_engine_rmse', 'mean_error'],
    'source': 'predictions_XGBoost.csv per split',
    'model_retrained': False,
}

for seed in [21, 84]:
    rv_dir = f'{RV_DIR}/split_seed_{seed}'
    pred_path    = f'{rv_dir}/predictions_XGBoost.csv'
    metrics_path = f'{rv_dir}/model_metrics.csv'
    manifest_path = f'{rv_dir}/run_manifest.json'

    pred_df = pd.read_csv(pred_path)
    per_eng = compute_per_engine_metrics(
        pred_df['RUL_capped'].values, pred_df['prediction'].values,
        pred_df[['unit_number', 'time_in_cycles']]
    )
    macro_rmse = round(float(per_eng['engine_rmse'].mean()), 6)
    mean_err   = round(float((pred_df['prediction'] - pred_df['RUL_capped']).mean()), 6)

    # Patch model_metrics.csv
    metrics_df = pd.read_csv(metrics_path)
    mask = metrics_df['model'] == 'XGBoost'
    metrics_df.loc[mask, 'macro_engine_rmse'] = macro_rmse
    metrics_df.loc[mask, 'mean_error']        = mean_err
    metrics_df.to_csv(metrics_path, index=False)

    # Patch run_manifest.json
    with open(manifest_path) as f:
        manifest = json.load(f)
    manifest['results']['XGBoost']['macro_engine_rmse'] = macro_rmse
    manifest['results']['XGBoost']['mean_error']        = mean_err
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)

    repair_log['seeds_repaired'].append({
        'split_seed':        seed,
        'macro_engine_rmse': macro_rmse,
        'mean_error':        mean_err,
    })
    print(f'  Seed {seed}: macro_engine_rmse={macro_rmse:.4f}  mean_error={mean_err:.4f}  — PATCHED')

repair_log_path = f'{RV_DIR}/xgboost_repair_log.json'
with open(repair_log_path, 'w') as f:
    json.dump(repair_log, f, indent=2)

print(f'
Repair log saved: {repair_log_path}')
print('XGBoost artefact repair complete. No models were retrained.')

### Split Seed 42 — Final Protocol

Seeds 21 and 84 artefacts are complete. I now run seed 42 under the same common protocol used for seeds 21 and 84 — the same `run_split_experiment()` function with per-model `clear_session` + `set_random_seed` resets.

The historical seed-42 models (from NB05/NB06) remain untouched in their original locations. Any difference in metrics between the historical and final-protocol seed-42 runs is expected: the final protocol resets the model random seed independently before each model, whereas the original NB05/NB06 notebooks set the seed once at notebook start in a different model construction context.

The two seed-42 result sets serve different purposes:

| Seed-42 result | Role |
|---|---|
| Historical NB05/NB06 | Mid-semester traceability and reproduction gate |
| Final-protocol NB10 | Repeated-validation comparison and NB11 epoch selection |

In [ ]:
results_42_final = run_split_experiment(42)

In [ ]:
# ── Seed-42 final-protocol artefact verification ────────────────────────────
print('Verifying seed-42 final-protocol artefacts...\n')

SEED42_DATA   = f'{FV_BASE}/split_seed_42'
SEED42_MODELS = f'{MV_BASE}/split_seed_42'
SEED42_RPTS   = f'{RV_DIR}/split_seed_42'

expected_files_42 = {
    'data': [
        f'{SEED42_DATA}/split_assignments.csv',
        f'{SEED42_DATA}/feature_lists.json',
        f'{SEED42_DATA}/scaler_b.joblib',
        f'{SEED42_DATA}/scaler_c.joblib',
    ],
    'models': [
        f'{SEED42_MODELS}/XGBoost_C.joblib',
        f'{SEED42_MODELS}/GRU_B_window30.keras',
        f'{SEED42_MODELS}/DerivedOnlyMLP_window30.keras',
        f'{SEED42_MODELS}/MultiViewGRUFusion_window30.keras',
    ],
    'reports': [
        f'{SEED42_RPTS}/model_metrics.csv',
        f'{SEED42_RPTS}/per_engine_metrics.csv',
        f'{SEED42_RPTS}/predictions_XGBoost.csv',
        f'{SEED42_RPTS}/predictions_GRU.csv',
        f'{SEED42_RPTS}/predictions_DerivedOnlyMLP.csv',
        f'{SEED42_RPTS}/predictions_MultiViewGRUFusion.csv',
        f'{SEED42_RPTS}/training_history_GRU.csv',
        f'{SEED42_RPTS}/training_history_DerivedOnlyMLP.csv',
        f'{SEED42_RPTS}/training_history_MultiViewGRUFusion.csv',
        f'{SEED42_RPTS}/run_manifest.json',
    ],
}

all_present_42 = True
for category, paths in expected_files_42.items():
    print(f'{category}:')
    for p in paths:
        exists = os.path.isfile(p)
        size_kb = os.path.getsize(p) / 1024 if exists else 0
        status = f'OK ({size_kb:.1f} KB)' if exists else 'MISSING'
        print(f'  {"OK" if exists else "!!"} {os.path.basename(p)} — {status}')
        if not exists:
            all_present_42 = False

assert all_present_42, 'One or more seed-42 final-protocol artefacts are missing.'

with open(f'{SEED42_RPTS}/run_manifest.json') as f:
    m42_final = json.load(f)
print(f'\nSeed-42 final-protocol manifest:')
print(f'  completed_at:    {m42_final["completed_at"]}')
print(f'  n_train_engines: {m42_final["n_train_engines"]}')
print(f'  n_val_engines:   {m42_final["n_val_engines"]}')
print(f'  n_train_windows: {m42_final["n_train_windows"]}')
print(f'  n_val_windows:   {m42_final["n_val_windows"]}')
print(f'  Results:')
for model_name, m in m42_final['results'].items():
    print(f'    {model_name:<25}  RMSE={m["rmse"]:.4f}  MAE={m["mae"]:.4f}  R2={m["r2"]:.4f}')

print('\nSeed-42 final-protocol artefact check: PASS — proceed to consolidation.')

### Consolidation: Seeds 21, 42, 84 (Final Protocol)

All three final-protocol splits are complete. I now consolidate results using only the common-protocol runs — seeds 21, 42, and 84 all trained via `run_split_experiment()`. The historical seed-42 result is retained in its original location for traceability but is not included in this comparison table.

The consolidation produces:
1. `repeated_validation_split_metrics_fd001.csv` — one row per (split_seed, model), full-protocol runs
2. `repeated_validation_summary_fd001.csv` — mean ± std across 3 splits, one row per model, includes macro-engine RMSE
3. `repeated_validation_pairwise_comparison_fd001.csv` — pairwise RMSE differences + fusion % comparisons + win counts
4. `per_engine_validation_metrics_fd001.csv` — combined per-engine metrics from all three splits

Note: mean and standard deviation across three splits are descriptive stability indicators, not strong statistical inference.

In [ ]:
# ── Load all three final-protocol split metrics ────────────────────────────
df_21 = pd.read_csv(f'{RV_DIR}/split_seed_21/model_metrics.csv')
df_42 = pd.read_csv(f'{RV_DIR}/split_seed_42/model_metrics.csv')
df_84 = pd.read_csv(f'{RV_DIR}/split_seed_84/model_metrics.csv')

# ── 1. Combined split metrics ──────────────────────────────────────────────
split_metrics = pd.concat([df_21, df_42, df_84], ignore_index=True)
split_metrics = split_metrics.sort_values(['model', 'split_seed']).reset_index(drop=True)
split_metrics_path = f'{RV_DIR}/repeated_validation_split_metrics_fd001.csv'
split_metrics.to_csv(split_metrics_path, index=False)
print('Split metrics (final-protocol, seeds 21/42/84):')
print(split_metrics[['split_seed','model','validation_rmse','validation_mae',
                      'validation_r2','macro_engine_rmse','mean_error']].to_string(index=False))

# ── 2. Summary: mean ± std across splits ──────────────────────────────────
model_order = ['MultiViewGRUFusion', 'XGBoost', 'DerivedOnlyMLP', 'GRU']
summary_rows = []
for model_name in model_order:
    sub = split_metrics[split_metrics['model'] == model_name]
    rmse_vals = sub['validation_rmse'].values
    mae_vals  = sub['validation_mae'].values
    r2_vals   = sub['validation_r2'].values
    macro_vals = sub['macro_engine_rmse'].values.astype(float)
    me_vals    = sub['mean_error'].values.astype(float)
    epochs     = sub['best_epoch'].dropna().values.tolist()
    summary_rows.append({
        'model':               model_name,
        'model_seed':          MODEL_SEED,
        'n_splits':            len(sub),
        'mean_rmse':           round(float(rmse_vals.mean()), 4),
        'std_rmse':            round(float(rmse_vals.std(ddof=1)), 4),
        'min_rmse':            round(float(rmse_vals.min()), 4),
        'max_rmse':            round(float(rmse_vals.max()), 4),
        'mean_mae':            round(float(mae_vals.mean()), 4),
        'std_mae':             round(float(mae_vals.std(ddof=1)), 4),
        'mean_r2':             round(float(r2_vals.mean()), 4),
        'std_r2':              round(float(r2_vals.std(ddof=1)), 4),
        'mean_macro_eng_rmse': round(float(macro_vals.mean()), 4),
        'std_macro_eng_rmse':  round(float(macro_vals.std(ddof=1)), 4),
        'mean_error_mean':     round(float(me_vals.mean()), 4),
        'mean_error_std':      round(float(me_vals.std(ddof=1)), 4),
        'best_epochs':         epochs,
    })
summary_df = pd.DataFrame(summary_rows)
summary_path = f'{RV_DIR}/repeated_validation_summary_fd001.csv'
summary_df.to_csv(summary_path, index=False)
print('\nSummary (mean ± std across 3 final-protocol splits):')
print(f'  {"Model":<25}  {"RMSE mean±std":>18}  {"MAE mean±std":>16}  {"R² mean":>7}  {"MacroEng mean±std":>20}  {"MeanErr mean":>12}')
for _, row in summary_df.iterrows():
    print(f'  {row["model"]:<25}  {row["mean_rmse"]:.4f} ± {row["std_rmse"]:.4f}  '
          f'{row["mean_mae"]:.4f} ± {row["std_mae"]:.4f}  '
          f'{row["mean_r2"]:.4f}  '
          f'{row["mean_macro_eng_rmse"]:.4f} ± {row["std_macro_eng_rmse"]:.4f}  '
          f'{row["mean_error_mean"]:+.4f}')
print('  Note: mean_error = prediction − actual; negative = under-prediction.')

# ── 3. Pairwise RMSE comparison + fusion comparisons ──────────────────────
pairwise_rows = []
for seed in SPLIT_SEEDS:
    sub = split_metrics[split_metrics['split_seed'] == seed].set_index('model')
    fusion_rmse = float(sub.loc['MultiViewGRUFusion', 'validation_rmse'])
    for m1 in model_order:
        for m2 in model_order:
            if m1 >= m2:
                continue
            rmse_a = float(sub.loc[m1, 'validation_rmse'])
            rmse_b = float(sub.loc[m2, 'validation_rmse'])
            pairwise_rows.append({
                'split_seed':          seed,
                'model_a':             m1,
                'model_b':             m2,
                'rmse_a':              round(rmse_a, 4),
                'rmse_b':              round(rmse_b, 4),
                'rmse_diff_a_minus_b': round(rmse_a - rmse_b, 4),
            })
pairwise_df = pd.DataFrame(pairwise_rows)
pairwise_path = f'{RV_DIR}/repeated_validation_pairwise_comparison_fd001.csv'
pairwise_df.to_csv(pairwise_path, index=False)
print(f'\nPairwise comparison saved: {len(pairwise_df)} rows')

# ── Fusion focused comparison ──────────────────────────────────────────────
comparators = ['XGBoost', 'GRU', 'DerivedOnlyMLP']
fusion_rows = []
for seed in SPLIT_SEEDS:
    sub = split_metrics[split_metrics['split_seed'] == seed].set_index('model')
    fusion_rmse = float(sub.loc['MultiViewGRUFusion', 'validation_rmse'])
    row = {'split_seed': seed, 'fusion_rmse': round(fusion_rmse, 4)}
    for cmp in comparators:
        cmp_rmse = float(sub.loc[cmp, 'validation_rmse'])
        pct = round((cmp_rmse - fusion_rmse) / cmp_rmse * 100, 2)
        row[f'{cmp}_rmse']  = round(cmp_rmse, 4)
        row[f'fusion_vs_{cmp}_pct'] = pct
    fusion_rows.append(row)
fusion_cmp_df = pd.DataFrame(fusion_rows)

# Win counts: fusion wins when its RMSE is strictly lower
for cmp in comparators:
    wins = sum(1 for _, r in fusion_cmp_df.iterrows() if r['fusion_rmse'] < r[f'{cmp}_rmse'])
    fusion_cmp_df[f'fusion_wins_vs_{cmp}'] = wins

fusion_cmp_path = f'{RV_DIR}/repeated_validation_fusion_comparison_fd001.csv'
fusion_cmp_df.to_csv(fusion_cmp_path, index=False)
print('\nFusion comparison:')
print(fusion_cmp_df[['split_seed','fusion_rmse','XGBoost_rmse','fusion_vs_XGBoost_pct',
                      'GRU_rmse','fusion_vs_GRU_pct',
                      'DerivedOnlyMLP_rmse','fusion_vs_DerivedOnlyMLP_pct']].to_string(index=False))
for cmp in comparators:
    wins = fusion_cmp_df[f'fusion_wins_vs_{cmp}'].iloc[0]
    print(f'  Fusion wins vs {cmp}: {wins}/3')

# ── 4. Combined per-engine metrics ────────────────────────────────────────
per_eng_21 = pd.read_csv(f'{RV_DIR}/split_seed_21/per_engine_metrics.csv')
per_eng_42 = pd.read_csv(f'{RV_DIR}/split_seed_42/per_engine_metrics.csv')
per_eng_84 = pd.read_csv(f'{RV_DIR}/split_seed_84/per_engine_metrics.csv')

per_engine_all = pd.concat([per_eng_21, per_eng_42, per_eng_84], ignore_index=True)
per_engine_all = per_engine_all.sort_values(['model','split_seed','unit_number']).reset_index(drop=True)
per_engine_path = f'{RV_DIR}/per_engine_validation_metrics_fd001.csv'
per_engine_all.to_csv(per_engine_path, index=False)
print(f'\nPer-engine metrics saved: {len(per_engine_all)} rows')

print('\n=== CONSOLIDATION COMPLETE ===')

### cycle_index Ablation (Seed 42 — Final Protocol)

I run the `cycle_index` ablation using the seed-42 final-protocol run as the with-`cycle_index` control baseline. This produces a cleaner paired comparison than using the historical NB06 result, because both the with- and without-`cycle_index` models are trained under the identical protocol.

GRU is excluded: it operates on raw sensor sequences and does not use the derived feature set.

The results are diagnostic only and do not update the main validation table. Removing `cycle_index` measures how much predictive value each model obtains from explicit lifecycle progression. This does not establish leakage, because cycle number is available at prediction time.

In [ ]:
print('=== cycle_index ABLATION (seed 42, final protocol) ===\n')

# ── Build ablation feature sets: remove cycle_index from derived cols ──────
derived_cols_no_ci = [c for c in DERIVED_COLS if c != 'cycle_index']
feature_set_c_no_ci = SENSOR_COLS + derived_cols_no_ci
assert 'cycle_index' not in derived_cols_no_ci
assert len(derived_cols_no_ci) == 42, f'Expected 42, got {len(derived_cols_no_ci)}'

# Recompute scaled Feature Set C for seed-42 without cycle_index
train_c_noci = train_c_raw_42.copy()
val_c_noci   = val_c_raw_42.copy()

from sklearn.preprocessing import StandardScaler as _SS
scaler_c_noci = _SS().fit(train_c_noci[feature_set_c_no_ci])
train_c_noci_scaled = train_c_noci.copy()
val_c_noci_scaled   = val_c_noci.copy()
train_c_noci_scaled[feature_set_c_no_ci] = scaler_c_noci.transform(train_c_noci[feature_set_c_no_ci])
val_c_noci_scaled[feature_set_c_no_ci]   = scaler_c_noci.transform(val_c_noci[feature_set_c_no_ci])

X_seq_tr_abl, X_der_tr_abl, y_tr_abl, meta_tr_abl = create_multiview_windows(
    train_b_42, train_c_noci_scaled, SENSOR_COLS, derived_cols_no_ci, TARGET_COL)
X_seq_val_abl, X_der_val_abl, y_val_abl, meta_val_abl = create_multiview_windows(
    val_b_42, val_c_noci_scaled, SENSOR_COLS, derived_cols_no_ci, TARGET_COL)

print(f'Ablation windows — train: {len(y_tr_abl)}  val: {len(y_val_abl)}')
assert len(y_val_abl) == len(y_42), 'Ablation val window count differs from gate run'

ablation_results = {}

# ── XGBoost ablation ───────────────────────────────────────────────────────
print('\n[XGBoost] ablation (no cycle_index)...')
xgb_abl = XGBRegressor(**XGB_PARAMS)
xgb_abl.fit(train_c_noci_scaled[feature_set_c_no_ci], train_c_noci_scaled[TARGET_COL])
val_c_noci_aligned = val_c_noci_scaled.merge(
    meta_val_abl[['unit_number','time_in_cycles']],
    on=['unit_number','time_in_cycles'], how='inner'
)
xgb_abl_pred = np.clip(xgb_abl.predict(val_c_noci_aligned[feature_set_c_no_ci]), 0, RUL_CAP)
xgb_abl_m = evaluate_predictions(val_c_noci_aligned[TARGET_COL].values, xgb_abl_pred)
ablation_results['XGBoost'] = xgb_abl_m
print(f'  RMSE={xgb_abl_m["rmse"]:.4f}  MAE={xgb_abl_m["mae"]:.4f}  R2={xgb_abl_m["r2"]:.4f}')

# ── DerivedOnlyMLP ablation ────────────────────────────────────────────────
print('\n[DerivedOnlyMLP] ablation (no cycle_index)...')
tf.keras.backend.clear_session(); gc.collect()
tf.keras.utils.set_random_seed(MODEL_SEED)
mlp_abl = build_derived_mlp(len(derived_cols_no_ci))
mlp_abl.fit(X_der_tr_abl, y_tr_abl,
            validation_data=(X_der_val_abl, y_val_abl),
            epochs=MLP_MAX_EPOCHS, batch_size=MLP_BATCH,
            callbacks=get_mlp_callbacks(), verbose=0)
mlp_abl_pred = np.clip(mlp_abl.predict(X_der_val_abl, verbose=0).ravel(), 0, RUL_CAP)
mlp_abl_m = evaluate_predictions(y_val_abl, mlp_abl_pred)
ablation_results['DerivedOnlyMLP'] = mlp_abl_m
print(f'  RMSE={mlp_abl_m["rmse"]:.4f}  MAE={mlp_abl_m["mae"]:.4f}  R2={mlp_abl_m["r2"]:.4f}')

# ── MultiViewGRUFusion ablation ────────────────────────────────────────────
print('\n[MultiViewGRUFusion] ablation (no cycle_index)...')
tf.keras.backend.clear_session(); gc.collect()
tf.keras.utils.set_random_seed(MODEL_SEED)
fusion_abl = build_multiview_gru(WINDOW_SIZE, len(SENSOR_COLS), len(derived_cols_no_ci))
fusion_abl.fit([X_seq_tr_abl, X_der_tr_abl], y_tr_abl,
               validation_data=([X_seq_val_abl, X_der_val_abl], y_val_abl),
               epochs=MLP_MAX_EPOCHS, batch_size=MLP_BATCH,
               callbacks=get_mlp_callbacks(), verbose=0)
fusion_abl_pred = np.clip(fusion_abl.predict([X_seq_val_abl, X_der_val_abl], verbose=0).ravel(), 0, RUL_CAP)
fusion_abl_m = evaluate_predictions(y_val_abl, fusion_abl_pred)
ablation_results['MultiViewGRUFusion'] = fusion_abl_m
print(f'  RMSE={fusion_abl_m["rmse"]:.4f}  MAE={fusion_abl_m["mae"]:.4f}  R2={fusion_abl_m["r2"]:.4f}')

# ── Load final-protocol seed-42 with-cycle_index controls ─────────────────
seed42_fp_metrics = pd.read_csv(f'{RV_DIR}/split_seed_42/model_metrics.csv').set_index('model')

abl_rows = []
for model_name in ['MultiViewGRUFusion', 'XGBoost', 'DerivedOnlyMLP']:
    rmse_with    = float(seed42_fp_metrics.loc[model_name, 'validation_rmse'])
    mae_with     = float(seed42_fp_metrics.loc[model_name, 'validation_mae'])
    r2_with      = float(seed42_fp_metrics.loc[model_name, 'validation_r2'])
    rmse_without = ablation_results[model_name]['rmse']
    mae_without  = ablation_results[model_name]['mae']
    r2_without   = ablation_results[model_name]['r2']
    abl_rows.append({
        'split_seed':      42,
        'model':           model_name,
        'rmse_with_ci':    round(rmse_with,    4),
        'rmse_without_ci': round(rmse_without, 4),
        'rmse_delta':      round(rmse_without - rmse_with, 4),
        'rmse_delta_pct':  round((rmse_without - rmse_with) / rmse_with * 100, 2),
        'mae_with_ci':     round(mae_with,     4),
        'mae_without_ci':  round(mae_without,  4),
        'mae_delta':       round(mae_without  - mae_with,  4),
        'r2_with_ci':      round(r2_with,      4),
        'r2_without_ci':   round(r2_without,   4),
        'r2_delta':        round(r2_without   - r2_with,   4),
        'control_source':  'final-protocol seed 42',
    })
ablation_df = pd.DataFrame(abl_rows)
ablation_path = f'{RV_DIR}/cycle_index_ablation_fd001.csv'
ablation_df.to_csv(ablation_path, index=False)

print('\ncycle_index ablation results (final-protocol seed 42 as control):')
print(f'  {"Model":<25}  {"RMSE+CI":>8}  {"RMSE-CI":>8}  {"ΔRMSE":>7}  {"ΔRMSE%":>7}  {"MAE+CI":>7}  {"MAE-CI":>7}  {"ΔMAE":>7}')
for _, row in ablation_df.iterrows():
    print(f'  {row["model"]:<25}  {row["rmse_with_ci"]:8.4f}  {row["rmse_without_ci"]:8.4f}  '
          f'{row["rmse_delta"]:+7.4f}  {row["rmse_delta_pct"]:+6.2f}%  '
          f'{row["mae_with_ci"]:7.4f}  {row["mae_without_ci"]:7.4f}  {row["mae_delta"]:+7.4f}')
print('  Positive Δ = degradation when cycle_index removed.')
print('  Interpretation: removing cycle_index measures the contribution of explicit lifecycle')
print('  progression to prediction. It does not establish leakage because cycle number is')
print('  available at prediction time.')
print(f'\nAblation table saved: {ablation_path}')
print('\n=== cycle_index ABLATION COMPLETE ===')

### Outcome Classification

Based on the three final-protocol split results, I classify the experimental outcome using the pre-defined scenarios from the frozen plan.

**Mean error convention:** mean error = prediction − actual. A negative value indicates systematic under-prediction.

#### Repeated-validation outcome

The repeated-validation summary shows that all four models achieve similar mean RMSE across three split configurations, with rankings varying by split. The fusion model (MultiViewGRUFusion) does not consistently outperform the comparators. DerivedOnlyMLP has lower standard deviation across splits, lower mean MAE, and leads on two of the three splits. The fusion model leads on one split.

> **Primary outcome: C — Fusion advantage is not consistently reproduced across split configurations.**
>
> **Secondary observation: D — Model rankings vary across engine-level split configurations, suggesting that split composition affects relative performance.**

This means the mid-semester seed-42 result, where MultiViewGRUFusion was the top model, was not replicated when the engine cohort was varied. The multi-view design does not demonstrate consistent generalisation advantage over a single-view derived-feature model on FD001.

#### Bias observation

MultiViewGRUFusion shows the strongest systematic under-prediction tendency across all three splits. Mean error is negative in all three runs, indicating the fusion model consistently under-estimates RUL. This is a directional bias not captured by RMSE alone and is worth noting in the discussion.

#### cycle_index contribution

Removing `cycle_index` degrades RMSE for all three affected models, with percentage degradation between approximately 12–20%. This indicates that explicit lifecycle position information contributes materially to prediction. The result does not establish leakage; cycle number is available at prediction time. The correct interpretation is that the models use both degradation-derived sensor behaviour and explicit lifecycle progression.

#### NB11 epoch selection

The best epochs from the three final-protocol splits will be used to select the training duration for full-training in NB11. The median best epoch across seeds 21, 42, and 84 will be used per model.

### Complete Artefact Listing

All NB10 outputs: gate artefacts, per-split training outputs, consolidation tables, and cycle_index ablation results.

In [22]:
import os

print('=== NB10 GENERATED ARTEFACTS ===\n')

# Gate and top-level consolidated outputs
print('reports/final_validation/ (gate + consolidated):')
top_level = [f for f in sorted(os.listdir(RV_DIR))
             if os.path.isfile(f'{RV_DIR}/{f}') and not f.startswith('.')]
for f in top_level:
    size_kb = os.path.getsize(f'{RV_DIR}/{f}') / 1024
    print(f'  {f} ({size_kb:.1f} KB)')

# Per-split outputs
for seed in SPLIT_SEEDS:
    split_rpt = f'{RV_DIR}/split_seed_{seed}'
    split_mdl = f'{MV_BASE}/split_seed_{seed}'
    split_dat = f'{FV_BASE}/split_seed_{seed}'
    print(f'\nSplit seed {seed}:')
    for d, label in [(split_dat, 'data'), (split_mdl, 'models'), (split_rpt, 'reports')]:
        if os.path.isdir(d):
            files = sorted(f for f in os.listdir(d)
                           if os.path.isfile(f'{d}/{f}') and not f.startswith('.'))
            for f in files:
                size_kb = os.path.getsize(f'{d}/{f}') / 1024
                print(f'  [{label}] {f} ({size_kb:.1f} KB)')

print('\n=== NB10 COMPLETE ===')

=== NB10 GENERATED ARTEFACTS ===

reports/final_validation/ (gate + consolidated):
  cycle_index_ablation_fd001.csv (0.4 KB)
  derived_feature_names.json (1.5 KB)
  experiment_environment.json (0.4 KB)
  frozen_experiment_config.json (2.4 KB)
  per_engine_validation_metrics_fd001.csv (18.1 KB)
  repeated_validation_pairwise_comparison_fd001.csv (0.9 KB)
  repeated_validation_split_metrics_fd001.csv (0.9 KB)
  repeated_validation_summary_fd001.csv (0.4 KB)
  seed42_reproduction_gate.csv (0.9 KB)
  seed42_reproduction_manifest.json (4.3 KB)

Split seed 21:
  [data] feature_lists.json (3.7 KB)
  [data] scaler_b.joblib (1.5 KB)
  [data] scaler_c.joblib (3.7 KB)
  [data] split_assignments.csv (0.8 KB)
  [models] DerivedOnlyMLP_window30.keras (87.4 KB)
  [models] GRU_B_window30.keras (349.0 KB)
  [models] MultiViewGRUFusion_window30.keras (393.0 KB)
  [models] XGBoost_C.joblib (514.8 KB)
  [reports] model_metrics.csv (0.4 KB)
  [reports] per_engine_metrics.csv (6.1 KB)
  [reports] prediction